# GUPPY PIPELINE 

In [ ]:
# using the EPM code as starting point

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

# ==========================================================
# Automated EPM + Fiber Photometry pipeline
# Flexible to zone numbering & header differences
# ==========================================================

# ----------------------------------------------------------
# Main configuration
# ----------------------------------------------------------

main_dir = Path(".").resolve()
timepoints = ["Preinduction", "W1", "W2", "W3"]

#focus on preinduction for now to save time 
#timepoints = ["Preinduction"]
#timepoints = ["W3"]
#timepoints = ["Preinduction","W3"]

# Nose-point zone substrings for 3CT (FLEXIBLE MATCHING)
nosepoint_substrings = [
    "Social_interaction",
    "Novel_interaction",
]

# Fixed colors keyed by inner substring
zone_colors = {
    "Social_interaction": "#377EB8",   # blue
    "Novel_interaction": "#E41A1C",  # red
}


video_fps = 50.0
fp_fps = 50.0

figures_main_dir = main_dir / "Figures"
figures_main_dir.mkdir(parents=True, exist_ok=True)

# ==========================================================
# Helper functions
# ==========================================================

def _mouse_genotype_tag(mouse_id: str):
    id_clean = mouse_id.strip()
    if id_clean in {"372", "376", "423", "398", "400", "459"}:
        return "NE"
    if id_clean in {"374", "429", "461", "463", "402"}:
        return "WT"
    return None


def _read_export_and_get_fp_and_mouse_id(xl_path: Path):
    df = pd.read_excel(xl_path, header=None)
    key_col = df.iloc[:, 0].astype(str).str.strip().str.lower()

    # FP file
    match_fp = df[key_col == "fp file"]
    if match_fp.empty:
        raise ValueError(f"'FP file' not found in {xl_path.name}")
    fp_id = str(df.iloc[match_fp.index[0], 1]).strip()

    # Mouse ID
    #match_id = df[key_col == "id"]
    match_id = df[key_col == "id"]
    if match_id.empty:
        raise ValueError(f"'ID' row not found in {xl_path.name}")
    mouse_id = str(df.iloc[match_id.index[0], 1]).strip()

    return fp_id, mouse_id

def _extract_condition(eth_path: Path) -> str:
    """
    Find the 'Condition' row in the metadata (top of the EthoVision file)
    and return its value (e.g. Hab, Social, Novel).
    """
    raw = pd.read_excel(eth_path, header=None, nrows=50, engine="openpyxl")
    for i in range(len(raw)):
        if str(raw.iloc[i, 0]).strip() == "Condition":
            cond = str(raw.iloc[i, 1]).strip()
            # Normalise a bit
            cond_norm = cond.capitalize()
            if cond_norm in {"Hab", "Social", "Novel"}:
                return cond_norm
            return cond
    return "Unknown"



def _find_fp_and_timestamp_files(fp_folder: Path, fp_id: str):
    fp_csv = None
    for p in fp_folder.glob("*.csv"):
        if fp_id.lower() in p.name.lower():
            #fp_csv = p
            fil = fp_id + '.csv'
            fp_csv = fp_folder / fil
            break
    if fp_csv is None:
        raise FileNotFoundError(f"No FP CSV found in {fp_folder}")

    ts_csv = None
    for p in fp_folder.glob("*.csv"):
        if "time" in p.name.lower() or "timestamp" in p.name.lower():
            #ts_csv = p
            import os
            directory, filename = os.path.split(p)

            # Remove leading "._" if present
            new_filename = filename.lstrip("._") if filename.startswith("._") else filename

            # Join back together
            new_path = os.path.join(directory, new_filename)
            
            ts_csv = new_path
            break
    if ts_csv is None:
        raise FileNotFoundError(f"No timestamp CSV found in {fp_folder}")

    return fp_csv, ts_csv


def _coerce_bool_col(series):
    if series.dtype == bool:
        return series
    if pd.api.types.is_numeric_dtype(series):
        return (series.astype(float) > 0).astype(bool)
    low = series.astype(str).str.strip().str.lower()
    return low.isin(["true", "1", "t", "yes", "y"])


def _get_video_window(ts: pd.DataFrame):
    """Robust detection of video ON/OFF timestamps."""
    # Time column
    tcol = next(c for c in ts.columns if "time" in c.lower())
    t = pd.to_numeric(ts[tcol], errors="coerce")

    # Digital channel
    scol = None
    for c in ts.columns:
        if "digital" in c.lower() or "state" in c.lower():
            scol = c
            break

    # No digital info → use full recording
    if scol is None:
        return float(t.iloc[0]), float(t.iloc[-1])

    state = _coerce_bool_col(ts[scol])

    # If no transitions → full interval
    if state.sum() == 0 or (~state).sum() == 0:
        return float(t.iloc[0]), float(t.iloc[-1])

    # Use first True and first False
    try:
        video_start = float(t[state].iloc[0])
        video_stop = float(t[~state].iloc[0])
        if video_stop > video_start:
            return video_start, video_stop
    except:
        pass

    # Fallback
    return float(t.iloc[0]), float(t.iloc[-1])


def _build_time_vector_from_fp(fp_df):
    if "SystemTimestamp" in fp_df.columns:
        ts = pd.to_numeric(fp_df["SystemTimestamp"], errors="coerce")
        if ts.notna().sum() > len(ts) * 0.8:
            return ts.to_numpy()
    return np.arange(len(fp_df)) / fp_fps


def _snap_down_index(t, target):
    return max(0, np.searchsorted(t, target, side="right") - 1)


def _trim_fp_to_window(fp_df, start, stop):
    tt = _build_time_vector_from_fp(fp_df)
    if len(tt) != len(fp_df):
        n = min(len(tt), len(fp_df))
        tt = tt[:n]
        fp_df = fp_df.iloc[:n]

    i0 = _snap_down_index(tt, start)
    i1 = _snap_down_index(tt, stop)
    out = fp_df.iloc[i0:i1+1].copy()
    out["Time_video"] = tt[i0:i1+1] - tt[i0]
    return out


def _detect_header_row(eth_path: Path):
    preview = pd.read_excel(eth_path, header=None, nrows=50)

    firstcell = str(preview.iloc[0, 0])
    if "number of header lines" in firstcell.lower():
        m = re.findall(r"\d+", firstcell)
        if m:
            return int(m[0])

    col0 = preview.iloc[:, 0].astype(str)
    hits = col0[col0.str.contains("Trial time", case=False, na=False)]
    if not hits.empty:
        return hits.index[0]

    raise ValueError(f"Cannot detect header row in {eth_path.name}")


def _load_ethovision_sheet_with_targets(eth_path: Path, video_fps: float):
    header_row = _detect_header_row(eth_path)
    eth = pd.read_excel(eth_path, header=header_row).dropna(axis=1, how="all")

    # Time_s
    tcol = next((c for c in eth.columns if "time" in c.lower()), None)
    if tcol:
        eth["Time_s"] = pd.to_numeric(eth[tcol], errors="coerce")
    else:
        eth["Time_s"] = np.arange(len(eth)) / video_fps

    # Flexible matching of nose-point zone columns
    matched_cols = []
    for sub in nosepoint_substrings:
        for col in eth.columns:
            if sub in str(col):
                matched_cols.append(col)
                break

    if not matched_cols:
        raise ValueError(f"No nose-point zones found in {eth_path.name}")

    return eth[["Time_s"] + matched_cols].copy(), matched_cols


def _ethogram_segments(eth, beh_cols):
    segments = []
    for beh in beh_cols:
        mask = eth[beh] == 1
        if mask.sum() == 0:
            continue
        diff = mask.astype(int).diff().fillna(0)
        starts = eth.loc[diff == 1, "Time_s"]
        ends = eth.loc[diff == -1, "Time_s"]

        if mask.iloc[0]:
            starts = pd.concat([pd.Series([eth["Time_s"].iloc[0]]), starts])
        if mask.iloc[-1]:
            ends = pd.concat([ends, pd.Series([eth["Time_s"].iloc[-1]])])

        for s, e in zip(starts, ends):
            segments.append(
                {"Behavior": beh, "Start_time_s": float(s), "End_time_s": float(e),
                 "Duration_s": float(e - s)}
            )
    return pd.DataFrame(segments)


def _guess_zone_color(colname):
    for sub, color in zone_colors.items():
        if sub in colname:
            return color
    return "black"


def _plot_trial(fp_trim, eth_timeline, beh_cols, out_png, title_prefix):
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True,
                             gridspec_kw={"height_ratios": [2, 2, 1]})

    chan = "G0" if "G0" in fp_trim.columns else next(
        c for c in fp_trim.columns
        if pd.api.types.is_numeric_dtype(fp_trim[c])
        and c not in {"LedState", "SystemTimestamp", "Time_video", "Timestamp"}
    )

    # LED 1 – isosbestic
    fp1 = fp_trim[fp_trim["LedState"] == 1]
    axes[0].plot(fp1["Time_video"], fp1[chan], linewidth=0.8, color="gray")
    axes[0].set_title(f"{title_prefix}: LED 1 (isosbestic)")
    axes[0].grid(True)

    # LED 2 – GCaMP
    fp2 = fp_trim[fp_trim["LedState"] == 2]
    axes[1].plot(fp2["Time_video"], fp2[chan], linewidth=0.8, color="green")
    axes[1].set_title(f"{title_prefix}: LED 2 (GCaMP)")
    axes[1].grid(True)

    # Ethogram
    for i, beh in enumerate(beh_cols):
        color = _guess_zone_color(beh)
        subset = eth_timeline[eth_timeline["Behavior"] == beh]
        for _, row in subset.iterrows():
            axes[2].barh(i, row["Duration_s"], left=row["Start_time_s"],
                         height=0.6, color=color)

    axes[2].set_yticks(range(len(beh_cols)))
    axes[2].set_yticklabels(beh_cols)
    axes[2].invert_yaxis()
    axes[2].set_xlabel("Time (s)")
    axes[2].set_title("Time in zone (nose-point)")

    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close(fig)


def _extract_led_traces(fp_trim, chan):
    iso = fp_trim.loc[fp_trim["LedState"] == 1, ["Time_video", chan]].rename(
        columns={chan: f"{chan}_iso415"})
    gcamp = fp_trim.loc[fp_trim["LedState"] == 2, ["Time_video", chan]].rename(
        columns={chan: f"{chan}_gcamp470"})
    return iso.reset_index(drop=True), gcamp.reset_index(drop=True)

# ==========================================================
# Main loop with week/timepoint added
# ==========================================================

fp_traces = {}

for tp in timepoints:
    print(f"\n=== Processing timepoint: {tp} ===")
    tp_dir = main_dir / tp
    export_dir = tp_dir / "Export files"

    fig_dir_tp = figures_main_dir / tp
    fig_dir_raw = fig_dir_tp / "Raw_traces_ethograms"
    fig_dir_raw.mkdir(parents=True, exist_ok=True)

    trial_files = sorted(export_dir.glob("Raw data-*.xlsx"))
    if not trial_files:
        print(f"[WARN] No Raw data-*.xlsx files in {export_dir}")
        continue

    for eth_path in trial_files:
        try:
            # Condition (Hab / Social / Novel)
            condition = _extract_condition(eth_path)
            fp_id, mouse_id = _read_export_and_get_fp_and_mouse_id(eth_path)
            geno = _mouse_genotype_tag(mouse_id)

            fp_folder = tp_dir / fp_id
            fp_csv, ts_csv = _find_fp_and_timestamp_files(fp_folder, fp_id)

            # Load FP
            fp = pd.read_csv(fp_csv, encoding="utf-8")
            fp = fp[fp["LedState"].isin([1, 2])]

            # Align timestamps
            ts = pd.read_csv(ts_csv, encoding="utf-8")
            video_start, video_stop = _get_video_window(ts)
            fp_trim = _trim_fp_to_window(fp, video_start, video_stop)

            # Detect channel
            chan = "G0" if "G0" in fp_trim.columns else next(
                c for c in fp_trim.columns
                if pd.api.types.is_numeric_dtype(fp_trim[c])
                and c not in {"LedState", "SystemTimestamp", "Time_video", "Timestamp"}
            )

            iso_df, gcamp_df = _extract_led_traces(fp_trim, chan)

            trial_stub = eth_path.stem.replace("Raw data-", "").replace(" ", "_")
            mouse_label = f"ID{mouse_id}" + (f"_{geno}" if geno else "")
            file_stub = f"{mouse_label}__{fp_id}__{trial_stub}"

            # Load EthoVision
            eth, matched_cols = _load_ethovision_sheet_with_targets(eth_path, video_fps)
            timeline = _ethogram_segments(eth, matched_cols)

            # Store all info in fp_traces, now including the 'week' field
            fp_traces[f"{tp}__{file_stub}"] = {
                "iso": iso_df,
                "gcamp": gcamp_df,
                "mouse_id": mouse_id,
                "genotype": geno,
                "timeline": timeline,
                "beh_cols": matched_cols,
                "condition": condition,
                "week": tp  # <--- NEW: store the timepoint/week
            }

            # Figure directory: by week and condition
            fig_dir_tp_cond = figures_main_dir / tp / condition / "Raw_traces_ethograms"
            fig_dir_tp_cond.mkdir(parents=True, exist_ok=True)

            # Plot trial
            out_png = fig_dir_tp_cond / f"{file_stub}_zones.png"
            _plot_trial(fp_trim, timeline, matched_cols, out_png,
                        title_prefix=f"{mouse_label} | {fp_id} | {trial_stub}")

            print(f"✔ {eth_path.name} → {out_png.name}")

        except Exception as e:
            print(f"[WARN] [{tp}] {eth_path.name}: {e}")


In [ ]:
timeline = _ethogram_segments(eth, matched_cols)
timeline


In [ ]:
list(fp_traces.keys())[:3]
first_key = next(iter(fp_traces))
fp_traces[first_key]


In [ ]:
import pandas as pd

fp_traces_df = pd.DataFrame.from_dict(fp_traces, orient="index")
fp_traces_df.head()


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True,
                         gridspec_kw={"height_ratios": [2, 2, 1]})

# --- FP traces ---
chan = "G0" if "G0" in fp_trim.columns else next(
    c for c in fp_trim.columns
    if pd.api.types.is_numeric_dtype(fp_trim[c])
    and c not in {"LedState","SystemTimestamp","Time_video","Timestamp"}
)

# LED 1
fp1 = fp_trim[fp_trim["LedState"] == 1]
axes[0].plot(fp1["Time_video"], fp1[chan], linewidth=0.8, color="gray")
axes[0].set_title(f"{mouse_label} | {fp_id} | {trial_stub} : LED 1 (isosbestic)")
axes[0].grid(True)

# LED 2
fp2 = fp_trim[fp_trim["LedState"] == 2]
axes[1].plot(fp2["Time_video"], fp2[chan], linewidth=0.8, color="green")
axes[1].set_title(f"{mouse_label} | {fp_id} | {trial_stub} : LED 2 (GCaMP)")
axes[1].grid(True)

# 
## --- Original ethogram ---
# beh_cols_orig = timeline["Behavior"].unique()
# for i, beh in enumerate(beh_cols_orig):
#     color = _guess_zone_color(beh)
#     subset = timeline[timeline["Behavior"] == beh]
#     for _, row in subset.iterrows():
#         axes[2].barh(i, row["Duration_s"], left=row["Start_time_s"], height=0.6, color=color)

# axes[2].set_yticks(range(len(beh_cols_orig)))
# axes[2].set_yticklabels(beh_cols_orig)

behaviors_in_timeline = timeline["Behavior"].unique()

for i, beh in enumerate(behaviors_in_timeline):
    color = _guess_zone_color(beh)
    subset = timeline[timeline["Behavior"] == beh]
    for _, row in subset.iterrows():
        axes[2].barh(i, row["Duration_s"], left=row["Start_time_s"],
                     height=0.6, color=color)

axes[2].set_yticks(range(len(behaviors_in_timeline)))
axes[2].set_yticklabels(behaviors_in_timeline)


axes[2].invert_yaxis()
axes[2].set_xlabel("Time (s)")
axes[2].set_title("Original ethogram")

axes[2].invert_yaxis()
axes[2].set_xlabel("Time (s)")
axes[2].set_title("Original ethogram")

# --- Cleaned / collapsed ethogram ---
#beh_cols_clean = timeline_cleaned["Behavior"].unique()
#for i, beh in enumerate(beh_cols_clean):<
#    color = _guess_zone_color(beh)
#    subset = timeline_cleaned[timeline_cleaned["Behavior"] == beh]
#    for _, row in subset.iterrows():
#        axes[3].barh(i, row["Duration_s"], left=row["Start_time_s"], height=0.6, color=color)
#
#axes[3].set_yticks(range(len(beh_cols_clean)))
#axes[3].set_yticklabels(beh_cols_clean)
#axes[3].invert_yaxis()
#axes[3].set_xlabel("Time (s)")
#axes[3].set_title("Collapsed + cleaned ethogram")

plt.tight_layout()
plt.show()


In [ ]:
#timeline_filter = timeline[timeline['Behavior']  != 'In zone 2(Arena / Nose-point)'].sort_values('Start_time_s')
timeline_filter = timeline[~timeline['Behavior'].str.contains('Arena / Nose-point')].sort_values('Start_time_s')

timeline_filter

In [ ]:
import pandas as pd

def multi_zone_to_segments_fixed(eth, zone_keywords=None):
    """
    Converts multiple zone columns into a clean timeline of behavior segments.
    Any time not in a zone is labeled 'Other'.
    Preserves zone names as behaviors.

    Args:
        eth (pd.DataFrame): DataFrame with 'Time_s' and zone columns.
        zone_keywords (list, optional): Keywords to identify zone columns.
                                        Default matches 'Zone'.

    Returns:
        pd.DataFrame: Timeline with ['Behavior', 'Start_time_s', 'End_time_s', 'Duration_s']
    """
    if zone_keywords is None:
        zone_keywords = ["interaction"]

    # Find all zone columns
    zone_cols = [c for c in eth.columns if any(k in c for k in zone_keywords)]

    if not zone_cols:
        raise ValueError("No zone columns detected. Check your zone_keywords or column names.")

    # Melt to long format
    eth_long = eth.melt(id_vars="Time_s", value_vars=zone_cols,
                        var_name="Zone_col", value_name="Zone_active")

    # Ensure any active zone is marked; convert to boolean
    eth_long["Zone_active"] = eth_long["Zone_active"].fillna(0)
    eth_long["Zone_active"] = eth_long["Zone_active"].astype(bool)

    # Sort by time
    eth_long_sorted = eth_long.sort_values(["Time_s", "Zone_col"])

    timeline_rows = []
    last_behavior = None
    start_time = eth["Time_s"].iloc[0]

    # Iterate over time points
    for t, group in eth_long_sorted.groupby("Time_s"):
        active_zones = group.loc[group["Zone_active"], "Zone_col"].tolist()
        current_behavior = active_zones[0] if active_zones else "Other"

        if last_behavior is None:
            last_behavior = current_behavior
            start_time = t
        elif current_behavior != last_behavior:
            # Close previous segment
            timeline_rows.append({
                "Behavior": last_behavior,
                "Start_time_s": float(start_time),
                "End_time_s": float(t),
                "Duration_s": float(t - start_time)
            })
            last_behavior = current_behavior
            start_time = t

    # Add final segment
    timeline_rows.append({
        "Behavior": last_behavior,
        "Start_time_s": float(start_time),
        "End_time_s": float(eth["Time_s"].iloc[-1]),
        "Duration_s": float(eth["Time_s"].iloc[-1] - start_time)
    })

    return pd.DataFrame(timeline_rows)

# ------------------------
# Example usage
timeline = multi_zone_to_segments_fixed(eth)
timeline


In [ ]:
# def smoothing_kernel(data, grainularity = 0.1, kernel_size = 5):
#     """
#     Smooths timeline data using a kernel.

#     Args:
#         data (pd.DataFrame): Timeline data.
#         grainularity (float): The grainularity that the timeline data should be split up with (seconds).
#         kernel_size (int): The number of 'frames' (defined by the grainularity) the kernel looks ahead 
#         and behind when determining the new lable.

#     Returns:
#         pd.DataFrame: The timeline data reassembled with the new labels in the column 'kernel_label'.
#     """
#     timeline_exploded = pd.DataFrame({'timestamp' : [], 'label' : []})

#     #def explode_timeline(timeline, grainularity):
#     for _, row in data.iterrows():
#         # timestamps = np.arange(row['Start_time_s'], row['End_time_s'], grainularity)
#         timestamps = np.arange(row['Start_time_s'], row['End_time_s'] + grainularity, grainularity)

#         labels = row['Behavior'] #* len(timestamps)

#         timeline_exploded = pd.concat([timeline_exploded, 
#                                        pd.DataFrame({'timestamp' : timestamps, 'label' : labels})])

#     timeline_exploded = timeline_exploded.reset_index()

#     kernel_content = []

#     def most_common(lst):
#         return max(set(lst), key=lst.count)

#     new_labels = []

#     for i, row in timeline_exploded.iterrows():
#         if i < kernel_size:
#             kernel_content = timeline_exploded[ : i + kernel_size]['label']

#         elif i + kernel_size > len(timeline_exploded):
#             kernel_content = timeline_exploded[i - (kernel_size - 1) : ]['label']

#         else:
#             kernel_content = timeline_exploded[i - (kernel_size - 1) : i + kernel_size]['label']
        
#         new_labels.append(most_common(kernel_content.to_list()))

#     timeline_exploded['kernel_label'] = new_labels

#     current_label = ''

#     timeline_kernel_labels = pd.DataFrame()

#     # for i, row in timeline_exploded.iterrows():
#     #     if current_label == row['kernel_label']:
#     #         pass

#     #     elif current_label == '':
#     #         current_label = row['kernel_label']
#     #         Start_time_s = row['timestamp']

#     #     elif current_label != row['kernel_label']:
#     #         timeline_kernel_labels = pd.concat([timeline_kernel_labels,
#     #                                             pd.DataFrame({
#     #                                                 'Behavior' : [current_label],
#     #                                                 'Start_time_s' : [Start_time_s],
#     #                                                 'End_time_s' : row['timestamp'],
#     #                                                 'Duration_s' : row['timestamp'] - Start_time_s
#     #                                             })])
#     #         Start_time_s = row['timestamp']
#     #         current_label = row['kernel_label']
    
#     # After your existing loop that builds timeline_kernel_labels
#     for i, row in timeline_exploded.iterrows():
#         if current_label == row['kernel_label']:
#             pass
#         elif current_label == '':
#             current_label = row['kernel_label']
#             Start_time_s = row['timestamp']
#         elif current_label != row['kernel_label']:
#             timeline_kernel_labels = pd.concat([timeline_kernel_labels,
#                                                 pd.DataFrame({
#                                                     'Behavior' : [current_label],
#                                                     'Start_time_s' : [Start_time_s],
#                                                     'End_time_s' : row['timestamp'],
#                                                     'Duration_s' : row['timestamp'] - Start_time_s
#                                                 })])
#             Start_time_s = row['timestamp']
#             current_label = row['kernel_label']

#     # --- Append the final segment ---
#     if current_label != '':
#         timeline_kernel_labels = pd.concat([timeline_kernel_labels,
#                                             pd.DataFrame({
#                                                 'Behavior': [current_label],
#                                                 'Start_time_s': [Start_time_s],
#                                                 'End_time_s': [timeline_exploded['timestamp'].iloc[-1]],
#                                                 'Duration_s': [timeline_exploded['timestamp'].iloc[-1] - Start_time_s]
#                                             })])
#    # code for removing anything shorter than 1 s
#     # --- Append the final segment ---
#     if current_label != '':
#         timeline_kernel_labels = pd.concat([timeline_kernel_labels,
#                                             pd.DataFrame({
#                                                 'Behavior': [current_label],
#                                                 'Start_time_s': [Start_time_s],
#                                                 'End_time_s': [timeline_exploded['timestamp'].iloc[-1]],
#                                                 'Duration_s': [timeline_exploded['timestamp'].iloc[-1] - Start_time_s]
#                                             })])

#     # --- FILTER: Remove segments shorter than 1 second ---
#     timeline_kernel_labels = timeline_kernel_labels[timeline_kernel_labels['Duration_s'] >= 1.0].reset_index(drop=True)



#     return timeline_kernel_labels

# timeline_filter = timeline[timeline['Behavior']  != 'In zone(Arena / Nose-point)'].sort_values('Start_time_s')
# timeline_filter
# timeline_kernel_labels = smoothing_kernel(timeline_filter, grainularity = 0.1, kernel_size = 10) #when granulairty is 0.1, it says every frame represents 0.1 second. the smaller granulairty, the less the end pieces weigh in the decision to what behaviour is being conducted




# timeline_kernel_labels

In [ ]:
# fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True,
#                          gridspec_kw={"height_ratios": [2, 2, 1, 1]})

# # --- FP traces ---
# chan = "G0" if "G0" in fp_trim.columns else next(
#     c for c in fp_trim.columns
#     if pd.api.types.is_numeric_dtype(fp_trim[c])
#     and c not in {"LedState","SystemTimestamp","Time_video","Timestamp"}
# )

# # LED 1
# fp1 = fp_trim[fp_trim["LedState"] == 1]
# axes[0].plot(fp1["Time_video"], fp1[chan], linewidth=0.8, color="gray")
# axes[0].set_title(f"{mouse_label} | {fp_id} | {trial_stub} : LED 1 (isosbestic)")
# axes[0].grid(True)

# # LED 2
# fp2 = fp_trim[fp_trim["LedState"] == 2]
# axes[1].plot(fp2["Time_video"], fp2[chan], linewidth=0.8, color="green")
# axes[1].set_title(f"{mouse_label} | {fp_id} | {trial_stub} : LED 2 (GCaMP)")
# axes[1].grid(True)

# # --- Original ethogram ---
# beh_cols_orig = timeline["Behavior"].unique()
# for i, beh in enumerate(beh_cols_orig):
#     color = _guess_zone_color(beh)
#     subset = timeline[timeline["Behavior"] == beh]
#     for _, row in subset.iterrows():
#         axes[2].barh(i, row["Duration_s"], left=row["Start_time_s"], height=0.6, color=color)

# axes[2].set_yticks(range(len(beh_cols_orig)))
# axes[2].set_yticklabels(beh_cols_orig)
# axes[2].invert_yaxis()
# axes[2].set_xlabel("Time (s)")
# axes[2].set_title("Original ethogram")

# # --- Smoothed / kernel ethogram ---
# # --- Flexible list of behaviors for smoothed ethogram ---
# beh_cols_clean = timeline_kernel_labels["Behavior"].unique().tolist()

# # Optional: sort so that "Other" comes first
# if "Other" in beh_cols_clean:
#     beh_cols_clean.remove("Other")
#     beh_cols_clean = ["Other"] + beh_cols_clean

# # --- Plot smoothed ethogram ---
# for i, beh in enumerate(beh_cols_clean):
#     color = _guess_zone_color(beh)
#     subset = timeline_kernel_labels[timeline_kernel_labels["Behavior"] == beh]
#     for _, row in subset.iterrows():
#         axes[3].barh(i, row["Duration_s"], left=row["Start_time_s"], height=0.6, color=color)

# axes[3].set_yticks(range(len(beh_cols_clean)))
# axes[3].set_yticklabels(beh_cols_clean)
# axes[3].invert_yaxis()
# axes[3].set_xlabel("Time (s)")
# axes[3].set_title("Kernel smoothed ethogram")


# plt.tight_layout()
# plt.show()


In [ ]:
# Sanity check - number of entries into different zones using original ethogram
print("\nOriginal ethogram occurrences:")

# Get unique behaviors in the original timeline
behaviors_in_timeline = timeline["Behavior"].unique()

for beh in behaviors_in_timeline:
    subset = timeline[timeline["Behavior"] == beh]
    n_occurrences = len(subset)
    print(f"{beh}: {n_occurrences} instance(s)")

    # Optional: print detailed start/end/duration for each instance
    for idx, row in subset.iterrows():
        print(f"  Start: {row['Start_time_s']:.3f} s, "
              f"End: {row['End_time_s']:.3f} s, "
              f"Duration: {row['Duration_s']:.3f} s")

# generate the above plot for all trials


In [ ]:
import pandas as pd

# ---------------------------------------------------------
# Preprocessing: Fill gaps in timeline with "Other" (Arena removed)
# ---------------------------------------------------------
def fill_gaps_with_other(timeline_df, video_duration=300):
    """
    Converts a timeline DataFrame into a full timeline with gaps labeled as "Other",
    removes all 'Arena / Nose-point' behaviors, and ensures alternating segments
    covering the full video duration.

    Args:
        timeline_df (pd.DataFrame): Output from _ethogram_segments
                                     with columns ['Behavior', 'Start_time_s', 'End_time_s', 'Duration_s']
        video_duration (float): total duration of the video in seconds

    Returns:
        pd.DataFrame: Timeline with 'Other' segments filling any unassigned time,
                      Arena behaviors removed, and full coverage of video duration.
    """
    if timeline_df.empty:
        return pd.DataFrame([{
            "Behavior": "Other",
            "Start_time_s": 0.0,
            "End_time_s": video_duration,
            "Duration_s": video_duration
        }])

    # Remove Arena behaviors completely
    timeline_df = timeline_df[~timeline_df['Behavior'].str.contains('Arena / Nose-point', case=False)]

    timeline = []
    # Sort by start time
    timeline_df = timeline_df.sort_values('Start_time_s').reset_index(drop=True)

    last_end = 0.0

    for _, row in timeline_df.iterrows():
        start = float(row["Start_time_s"])
        end = float(row["End_time_s"])

        # Fill gap with Other if there is any
        if start > last_end:
            timeline.append({
                "Behavior": "Other",
                "Start_time_s": last_end,
                "End_time_s": start,
                "Duration_s": start - last_end
            })

        # Add current behavior
        timeline.append({
            "Behavior": row["Behavior"],
            "Start_time_s": start,
            "End_time_s": end,
            "Duration_s": end - start
        })

        last_end = end

    # Fill remaining time to end of video if necessary
    if last_end < video_duration:
        timeline.append({
            "Behavior": "Other",
            "Start_time_s": last_end,
            "End_time_s": video_duration,
            "Duration_s": video_duration - last_end
        })

    return pd.DataFrame(timeline).sort_values("Start_time_s").reset_index(drop=True)

for trial_key, trial_data in fp_traces.items():
    try:
        timeline = trial_data["timeline"]  # from _ethogram_segments
        timeline_full = fill_gaps_with_other(timeline, video_duration=300)
        trial_data["timeline_full"] = timeline_full
        trial_data["beh_cols_full"] = timeline_full["Behavior"].unique()
    except Exception as e:
        print(f"[WARN] Could not preprocess trial {trial_key}: {e}")

# Check first trial
first_trial_key = list(fp_traces.keys())[0]
print(fp_traces[first_trial_key]["timeline_full"].head(200))



In [ ]:
# Get first trial key
first_trial_key = list(fp_traces.keys())[0]
first_trial = fp_traces[first_trial_key]

# Access the full timeline with "Other"
timeline_full = first_trial["timeline_full"]

# Print the first 20 rows to check
print(timeline_full.head(200))


In [ ]:
# ==========================================================
# Plot original ethograms for all trials (no kernel smoothing)
# ==========================================================

for trial_key, trial_data in fp_traces.items():
    try:
        # --- Extract FP traces and original timeline ---
        iso_df = trial_data["iso"]
        gcamp_df = trial_data["gcamp"]
        timeline = trial_data["timeline"]  # Use original ethogram
        mouse_label = f"ID{trial_data['mouse_id']}" + (f"_{trial_data['genotype']}" if trial_data['genotype'] else "")
        fp_id = trial_key.split("__")[1]
        trial_stub = "__".join(trial_key.split("__")[2:])

        # --- Filter timeline (exclude Arena only if desired) ---
        timeline_filter = timeline[~timeline['Behavior'].str.contains('Arena / Nose-point', case=False)].sort_values('Start_time_s')

        # --- Setup 3-panel figure ---
        fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True,
                                 gridspec_kw={"height_ratios": [2, 2, 1]})

        # --- Panel 1: LED 1 (isosbestic) ---
        iso_col = [c for c in iso_df.columns if c != "Time_video"][0]
        axes[0].plot(iso_df["Time_video"], iso_df[iso_col], linewidth=0.8, color="gray")
        axes[0].set_title(f"{mouse_label} | {fp_id} | {trial_stub} : LED 1 (isosbestic)")
        axes[0].grid(True)

        # --- Panel 2: LED 2 (GCaMP) ---
        gcamp_col = [c for c in gcamp_df.columns if c != "Time_video"][0]
        axes[1].plot(gcamp_df["Time_video"], gcamp_df[gcamp_col], linewidth=0.8, color="green")
        axes[1].set_title(f"{mouse_label} | {fp_id} | {trial_stub} : LED 2 (GCaMP)")
        axes[1].grid(True)

        # --- Panel 3: Original ethogram ---
        beh_cols_orig = trial_data["beh_cols"]

        # Plot each behavior/zone
        for i, beh in enumerate(beh_cols_orig):
            color = _guess_zone_color(beh)
            subset = timeline_filter[timeline_filter["Behavior"] == beh]
            for _, row in subset.iterrows():
                axes[2].barh(i, row["Duration_s"], left=row["Start_time_s"], height=0.6, color=color)

        # Plot "Other" if present
        if "Other" in timeline_filter["Behavior"].values:
            i = len(beh_cols_orig)
            subset = timeline_filter[timeline_filter["Behavior"] == "Other"]
            for _, row in subset.iterrows():
                axes[2].barh(i, row["Duration_s"], left=row["Start_time_s"], height=0.6, color="#CCCCCC")
            axes[2].set_yticks(range(len(beh_cols_orig)+1))
            axes[2].set_yticklabels(list(beh_cols_orig) + ["Other"])
        else:
            axes[2].set_yticks(range(len(beh_cols_orig)))
            axes[2].set_yticklabels(beh_cols_orig)

        axes[2].invert_yaxis()
        axes[2].set_xlabel("Time (s)")
        axes[2].set_title("Original ethogram")

        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"[WARN] Could not plot trial {trial_key}: {e}")

In [ ]:
for trial_key, trial_data in fp_traces.items():
    print(f"\nTrial: {trial_key}")
    if "timeline_kernel" not in trial_data:
        print("  No kernel-smoothed ethogram found.")
        continue

    tk = trial_data["timeline_kernel"]

    # Count number of separate occurrences for each behavior
    counts = tk["Behavior"].value_counts()
    for beh, n in counts.items():
        print(f"  {beh}: {n} instance(s)")


# code for manually subtracting iso from gcamp by aligning iso and gcamp to occur at same timepoints (a bit artificial, they do not occur at exactly the same timepoint)

In [ ]:
# ==========================================================
# Coregister ISO and GCaMP traces for all timepoints
# ==========================================================

import pandas as pd

def coregister_iso_gcamp(trial_traces: dict) -> pd.DataFrame:
    """
    trial_traces: dict with keys 'iso' and 'gcamp', each a DataFrame like:
        iso:   ['Time_video', 'G0_iso415']
        gcamp:['Time_video', 'G0_gcamp470']

    Returns one DataFrame with:
        Time_video  (from the channel that starts first)
        iso415
        gcamp470
    """
    iso = trial_traces["iso"].copy()
    gcamp = trial_traces["gcamp"].copy()

    # Get signal column names
    iso_col   = [c for c in iso.columns   if c != "Time_video"][0]
    gcamp_col = [c for c in gcamp.columns if c != "Time_video"][0]

    # Trim to common length just in case
    n = min(len(iso), len(gcamp))
    iso   = iso.iloc[:n].reset_index(drop=True)
    gcamp = gcamp.iloc[:n].reset_index(drop=True)

    # Decide which channel is "main" based on starting time
    if gcamp["Time_video"].iloc[0] <= iso["Time_video"].iloc[0]:
        # gcamp starts first → use its time
        time_main = gcamp["Time_video"].values
    else:
        # iso starts first → use its time
        time_main = iso["Time_video"].values

    aligned = pd.DataFrame({
        "Time_video": time_main,
        "iso415": iso[iso_col].values,
        "gcamp470": gcamp[gcamp_col].values,
    })

    return aligned

# -------------------------------------------------
# Apply coregistration to all trials in fp_traces
# -------------------------------------------------

coregistered_traces = {}  # store results per trial

for tp in timepoints:
    # Filter keys for this timepoint
    tp_keys = [k for k in fp_traces.keys() if k.startswith(f"{tp}__")]
    for k in tp_keys:
        try:
            aligned_df = coregister_iso_gcamp(fp_traces[k])
            coregistered_traces[k] = aligned_df
        except Exception as e:
            print(f"[WARN] Could not coregister {k}: {e}")

# Example access:


In [ ]:
# Show the first key
first_key = list(coregistered_traces.keys())[1]
print("First trial key:", first_key)

# Show the head of the aligned dataframe
print(coregistered_traces[first_key].head())


# apply zero-phase moving window - not using subtracted data

After extraction of the data, a zero-phase moving average linear digital filter is applied backwards and forwards to both the channels to reduce high frequency noise without time-shifting. The standard window for the moving filter is 100 data points i GuPPy, but can be adjusted by the user when setting input parameters. The window needs to be adjusted depending on the sampling rate of the recorded data. Guidelines are provided in the user guide. The traces resulting after this filtering step will be displayed for the user to inspect before proceeding with analysis.

In [ ]:
import numpy as np
from scipy.signal import filtfilt

def zero_phase_moving_average(signal, window_pts=5):
    """
    GuPPy-style zero-phase moving average.
    """
    signal = np.asarray(signal, dtype=float)

    if window_pts < 1:
        raise ValueError("window_pts must be >= 1")

    kernel = np.ones(window_pts, dtype=float) / window_pts

    filtered = filtfilt(kernel, [1.0], signal, padlen=3 * window_pts)
    return filtered


# ==========================================================
# Apply zero-phase moving average to all aligned trials
# ==========================================================

smoothed_trials = {}

window_pts = 5  # ≈ 4 seconds at 25 Hz alternation

for key, df in coregistered_traces.items():     # FIXED HERE
    try:
        smoothed_df = df.copy()
        smoothed_df["iso415_smooth"]   = zero_phase_moving_average(df["iso415"].values, window_pts)
        smoothed_df["gcamp470_smooth"] = zero_phase_moving_average(df["gcamp470"].values, window_pts)
        smoothed_trials[key] = smoothed_df
    except Exception as e:
        print(f"[WARN] Could not smooth trial {key}: {e}")

# Example output
example_key = list(smoothed_trials.keys())[0]
print(f"Example smoothed trial: {example_key}")
print(smoothed_trials[example_key].head())


Nice, you found the key bit in their methods. That paragraph is describing what happens on the TDT rig, not in GuPPy itself:
465 nm and 405 nm LEDs … were demodulated using a 4 Hz low-pass frequency filter.
So there are actually two separate filtering stages in their setup:
Hardware / Synapse side (TDT)
LEDs modulated at ~200–330 Hz
Demodulated and low-pass filtered at 4 Hz on the RZ5P
This already strips out the fast carrier and most high-frequency noise
Software side (GuPPy)
GuPPy then adds a zero-phase moving average filter with a user-defined window (default 100 points) on the demodulated streams
The “Window for Moving Average filter” in GuPPy refers to step 2.
How to think about window_pts in terms of your sampling rate
​	
Larger M → lower cutoff → more aggressive smoothing
Smaller M → higher cutoff → lighter smoothing
What did GuPPy likely do in their own rig?
On TDT they say “demodulated using a 4 Hz low-pass filter”. If their demodulated stream was at around 1 kHz (typical for RZ5P), then:

​	
 ≈4.4 Hz
So a 100-point moving average at ~1 kHz naturally gives about a 4 Hz cutoff, which matches their description.
In your case, though, you’re working with already demultiplexed / downsampled FP3002-style traces at ~25 Hz per channel.
What is “appropriate” for 25 Hz per channel?

​	
 
Some example window_pts:
M (window_pts)	f_c (approx)	Smoothed time scale
3	~3.7 Hz	very light smoothing, almost raw
5	~2.2 Hz	light smoothing
11	~1.0 Hz	moderate smoothing over ~1 s
25	~0.44 Hz	strong smoothing over several seconds
75	~0.15 Hz	very slow trend only, kills fast dynamics

Start with window_pts = 5–11
5 points → f_c ≈ 2.2 Hz (light smoothing)
11 points → f_c ≈ 1.0 Hz (moderate smoothing)

what you’re plotting is not the regression line in 2D space, but the 405 time-series passed through the regression model.

In [ ]:
import numpy as np

preproc_trials = {}

for key, df in smoothed_trials.items():
    df = df.copy()

    # Take smoothed iso (control) and gcamp (signal)
    x = df["iso415_smooth"].to_numpy()
    y = df["gcamp470_smooth"].to_numpy()

    # Mask invalid values
    mask = np.isfinite(x) & np.isfinite(y)
    x_fit = x[mask]
    y_fit = y[mask]

    if len(x_fit) < 2:
        print(f"[WARN] Not enough valid points for trial {key}")
        continue

    # Linear regression: y ≈ a*x + b
    a, b = np.polyfit(x_fit, y_fit, 1)

    # Fitted baseline
    Y_fit_all = a * x + b

    # Delta F
    Y_dF_all = y - Y_fit_all

    # ∆F/F (%)
    eps = np.finfo(float).eps
    dFF = 100.0 * (Y_dF_all / (Y_fit_all + eps))

    # z-scored ∆F/F
    mu  = np.nanmean(dFF)
    sig = np.nanstd(dFF)
    dFF_z = (dFF - mu)/sig if sig > 0 else np.zeros_like(dFF)

    # Store in DataFrame
    df["iso_fit"] = Y_fit_all
    df["dFF"] = dFF
    df["dFF_z"] = dFF_z

    preproc_trials[key] = df


In [ ]:
import matplotlib.pyplot as plt

def plot_preproc_trial(df, key, save_dir=None):
    """
    Plot:
        1) ISO smoothed
        2) GCaMP smoothed
        3) Fitted baseline
        4) dF/F (%)

    df must contain:
        Time_video
        iso415_smooth
        gcamp470_smooth
        iso_fit
        dFF
    """
    time = df["Time_video"].values

    iso  = df["iso415_smooth"].values
    gcamp = df["gcamp470_smooth"].values
    baseline = df["iso_fit"].values
    dFF = df["dFF"].values

    fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

    # 1) ISO 415
    axes[0].plot(time, iso, color="gray", linewidth=1)
    axes[0].set_title(f"{key} — ISO 415 smoothed")
    axes[0].set_ylabel("Signal")

    # 2) GCaMP 470
    axes[1].plot(time, gcamp, color="green", linewidth=1)
    axes[1].set_title("GCaMP 470 smoothed")
    axes[1].set_ylabel("Signal")

    # 3) Baseline fit vs GCaMP
    axes[2].plot(time, gcamp, color="green", linewidth=0.8, label="GCaMP")
    axes[2].plot(time, baseline, color="black", linewidth=0.8, linestyle="--",
                 label="Fitted baseline")
    axes[2].set_title("GCaMP with fitted baseline")
    axes[2].legend(loc="upper right")
    axes[2].set_ylabel("Value")

    # 4) dF/F
    axes[3].plot(time, dFF, color="purple", linewidth=0.8)
    axes[3].set_title("dF/F (%)")
    axes[3].set_xlabel("Time (s)")
    axes[3].set_ylabel("%")

    plt.tight_layout()

    # Save or show
    if save_dir is not None:
        out = save_dir / f"{key}_preproc.png"
        plt.savefig(out, dpi=200)
        plt.close()
    else:
        plt.show()


In [ ]:
# Plot preprocessed trials with fitted baseline and dF/F
for key in preproc_trials.keys():  # <-- use preproc_trials, not smoothed_trials
    try:
        plot_preproc_trial(preproc_trials[key], key)
    except Exception as e:
        print(f"[WARN] Could not plot {key}: {e}")


# Putting together corrections to ethovision ethogram and FP preprocessed traces

below code uses all the preprocessed signals, also the intermediares + skip to the next block to see only delta F over F signal

In [ ]:
def map_ethogram_to_time_vector(time_vector, timeline_ethogram):
    """
    Assign behavior to each FP timepoint exactly as in the original ethogram.
    Fills any gaps with 'Other'.

    Args:
        time_vector (np.ndarray or pd.Series): Timepoints from FP recording.
        timeline_ethogram (pd.DataFrame): Original ethogram with columns
                                          ['Behavior', 'Start_time_s', 'End_time_s'].

    Returns:
        np.ndarray: Array of behaviors for each timepoint in `time_vector`.
    """
    behavior_at_time = np.array(["Other"] * len(time_vector), dtype=object)

    for _, row in timeline_ethogram.iterrows():
        mask = (time_vector >= row["Start_time_s"]) & (time_vector < row["End_time_s"])
        behavior_at_time[mask] = row["Behavior"]

    return behavior_at_time

In [ ]:


# combined_trials = {}

# for key in preproc_trials.keys():
#     df = preproc_trials[key].copy()
#     if "timeline_kernel" in fp_traces[key]:
#         timeline_kernel = fp_traces[key]["timeline_kernel"]
#         df["Behavior"] = map_ethogram_to_time_vector(df["Time_video"].values, timeline_kernel)
#     else:
#         df["Behavior"] = "Other"  # fallback
#     combined_trials[key] = df


combined_trials = {}

for key in preproc_trials.keys():
    df = preproc_trials[key].copy()
    
    # Use the original ethogram instead of kernel-smoothed
    if "timeline" in fp_traces[key]:
        timeline_orig = fp_traces[key]["timeline"]
        df["Behavior"] = map_ethogram_to_time_vector(df["Time_video"].values, timeline_orig)
    else:
        df["Behavior"] = "Other"  # fallback if timeline not present
    
    combined_trials[key] = df

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def plot_fp_with_ethogram(df, key, save_dir=None):
    """
    Plot FP preprocessed traces together with kernel-smoothed ethogram.
    
    df must contain:
        Time_video
        iso415_smooth
        gcamp470_smooth
        iso_fit
        dFF
        Behavior
    """
    time = df["Time_video"].values
    iso = df["iso415_smooth"].values
    gcamp = df["gcamp470_smooth"].values
    baseline = df["iso_fit"].values
    dFF = df["dFF"].values
    behaviors = df["Behavior"].values

    # Unique behaviors and colors
    beh_labels = np.unique(behaviors)
    colors = plt.cm.tab10(np.arange(len(beh_labels)))  # up to 10 colors
    beh_color_map = dict(zip(beh_labels, colors))

    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True,
                             gridspec_kw={"height_ratios": [2, 2, 1, 1]})

    # --- 1) ISO 415 ---
    axes[0].plot(time, iso, color="gray", linewidth=1)
    axes[0].set_ylabel("ISO 415")
    axes[0].set_title(f"{key} — FP + Kernel-smoothed ethogram")

    # --- 2) GCaMP 470 ---
    axes[1].plot(time, gcamp, color="green", linewidth=1)
    axes[1].set_ylabel("GCaMP 470")

    # --- 3) Baseline vs GCaMP ---
    axes[2].plot(time, gcamp, color="green", linewidth=0.8, label="GCaMP")
    axes[2].plot(time, baseline, color="black", linewidth=0.8, linestyle="--", label="Fitted baseline")
    axes[2].set_ylabel("Value")
    axes[2].legend(loc="upper right")

    # --- 4) dF/F (%) ---
    axes[3].plot(time, dFF, color="purple", linewidth=0.8)
    axes[3].set_ylabel("dF/F (%)")
    axes[3].set_xlabel("Time (s)")

    # --- Overlay ethogram as colored bands ---
    for ax in axes:
        for beh in beh_labels:
            mask = behaviors == beh
            ax.fill_between(time, ax.get_ylim()[0], ax.get_ylim()[1],
                            where=mask, color=beh_color_map[beh], alpha=0.2, step="post")

    # Create legend for behaviors
    handles = [mpatches.Patch(color=beh_color_map[b], alpha=0.3, label=b) for b in beh_labels]
    axes[0].legend(handles=handles, bbox_to_anchor=(1.05, 1), loc="upper left")

    plt.tight_layout()

    # Save or show
    if save_dir is not None:
        out = save_dir / f"{key}_FP_ethogram.png"
        plt.savefig(out, dpi=200)
        plt.close()
    else:
        plt.show()


In [ ]:
example_key = list(combined_trials.keys())[1]
plot_fp_with_ethogram(combined_trials[example_key], example_key)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def plot_fp_with_ethogram(df, key, save_dir=None):
    """
    Plot FP preprocessed traces together with the original ethogram.

    df must contain:
        Time_video
        iso415_smooth
        gcamp470_smooth
        iso_fit
        dFF
        Behavior  # mapped from original ethogram
    """
    time = df["Time_video"].values
    dFF = df["dFF"].values
    behaviors = df["Behavior"].values  # original ethogram

    # Unique behaviors and colors
    beh_labels = np.unique(behaviors)
    colors = plt.cm.tab10(np.arange(len(beh_labels)))  # up to 10 colors
    beh_color_map = dict(zip(beh_labels, colors))

    if save_dir is not None:
        # --- Create full 4-subplot figure for saving ---
        iso = df["iso415_smooth"].values
        gcamp = df["gcamp470_smooth"].values
        baseline = df["iso_fit"].values

        fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True,
                                 gridspec_kw={"height_ratios": [2, 2, 1, 1]})

        axes[0].plot(time, iso, color="gray", linewidth=1)
        axes[0].set_ylabel("ISO 415")
        axes[0].set_title(f"{key} — FP + Original ethogram")

        axes[1].plot(time, gcamp, color="green", linewidth=1)
        axes[1].set_ylabel("GCaMP 470")

        axes[2].plot(time, gcamp, color="green", linewidth=0.8, label="GCaMP")
        axes[2].plot(time, baseline, color="black", linewidth=0.8, linestyle="--", label="Fitted baseline")
        axes[2].set_ylabel("Value")
        axes[2].legend(loc="upper right")

        axes[3].plot(time, dFF, color="purple", linewidth=0.8)
        axes[3].set_ylabel("dF/F (%)")
        axes[3].set_xlabel("Time (s)")

        # Overlay ethogram (original)
        for ax in axes:
            for beh in beh_labels:
                mask = behaviors == beh
                ax.fill_between(time, ax.get_ylim()[0], ax.get_ylim()[1],
                                where=mask, color=beh_color_map[beh], alpha=0.2, step="post")

        # Legend for behaviors
        handles = [mpatches.Patch(color=beh_color_map[b], alpha=0.3, label=b) for b in beh_labels]
        axes[0].legend(handles=handles, bbox_to_anchor=(1.05, 1), loc="upper left")

        plt.tight_layout()
        if save_dir is not None:
            out = save_dir / f"{key}_FP_ethogram.png"
            plt.savefig(out, dpi=200)
            plt.close()

    else:
        # --- Only plot last subplot interactively ---
        plt.figure(figsize=(14, 3))
        plt.plot(time, dFF, color="purple", linewidth=0.8)
        plt.ylabel("dF/F (%)")
        plt.xlabel("Time (s)")
        plt.title(f"{key} — dF/F (%) + Original ethogram")

        # Overlay ethogram
        for beh in beh_labels:
            mask = behaviors == beh
            plt.fill_between(time, plt.ylim()[0], plt.ylim()[1],
                             where=mask, color=beh_color_map[beh], alpha=0.2, step="post")

        # Legend for behaviors
        handles = [mpatches.Patch(color=beh_color_map[b], alpha=0.3, label=b) for b in beh_labels]
        plt.legend(handles=handles, bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)

        plt.tight_layout()
        plt.show()

In [ ]:
# example_key = list(combined_trials.keys())[0]
# plot_fp_with_ethogram(combined_trials[example_key], example_key)

example_key = list(combined_trials.keys())[0]
plot_fp_with_ethogram(combined_trials[example_key], example_key)


In [ ]:
# Loop over all trials and display the dF/F plot with ethogram and legend
for example_key in combined_trials.keys():
    plot_fp_with_ethogram(combined_trials[example_key], example_key, save_dir=None)

# PERI EVENT PLOTS CORRECTED SIGNAL


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------- PARAMETERS ----------
pre_s = 2.0      # seconds before event
post_s = 4.0     # seconds after event
min_stay_s = 0   # optional, if you want to filter very short events

# Pick first trial
example_key = list(combined_trials.keys())[1]
df = combined_trials[example_key].copy()

time_vec = df["Time_video"].values
signal_vec = df["dFF"].values
behaviors = df["Behavior"].values

# Unique behaviors
unique_beh = np.unique(behaviors)

# Loop over behaviors
for beh in unique_beh:
    # Find event starts (transition into this behavior)
    mask = (behaviors == beh)
    mask_int = mask.astype(int)  # convert bool to int
    diffs = np.diff(np.r_[0, mask_int])  # prepend 0 for first element
    event_starts_idx = np.where(diffs == 1)[0]  # rising edges

    # Skip if no eventsimport numpy as np
import matplotlib.pyplot as plt

# ---------- PARAMETERS ----------
pre_s = 2.0      # seconds before event
post_s = 4.0     # seconds after event
min_stay_s = 0   # optional: ignore very short events

# Pick first example trial
example_key = list(combined_trials.keys())[1]
df = combined_trials[example_key].copy()

# FP time and signal
time_vec = df["Time_video"].values
signal_vec = df["dFF"].values

# Behavior labels from the **original ethogram**
behaviors = df["Behavior"].values
unique_beh = np.unique(behaviors)

# Loop over behaviors
for beh in unique_beh:
    # Detect event start indices (transitions into this behavior)
    mask = (behaviors == beh)
    diffs = np.diff(np.r_[0, mask.astype(int)])  # rising edges indicate event start
    event_starts_idx = np.where(diffs == 1)[0]

    if len(event_starts_idx) == 0:
        print(f"No events found for {beh}")
        continue

    # Convert pre/post window to sample indices
    fps = 1 / np.mean(np.diff(time_vec))  # effective sampling rate
    pre_n = int(pre_s * fps)
    post_n = int(post_s * fps)

    # Collect peri-event segments
    segments = []
    for idx in event_starts_idx:
        # Ensure window fits within signal
        if idx >= pre_n and (idx + post_n) < len(signal_vec):
            seg = signal_vec[idx-pre_n : idx+post_n].copy()
            seg = seg - np.mean(seg[:pre_n])  # baseline correction
            segments.append(seg)

    if len(segments) == 0:
        print(f"No valid peri-event segments for {beh}")
        continue

    # Stack segments into array
    ERF = np.column_stack(segments)
    t_window = np.linspace(-pre_s, post_s, pre_n + post_n)

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(8,4))
    # Individual traces
    colors = plt.cm.tab10(np.linspace(0,1,ERF.shape[1]))
    for i, col in enumerate(colors):
        ax.plot(t_window, ERF[:,i], color=col, alpha=0.3, lw=1)
    # Mean trace
    ax.plot(t_window, ERF.mean(axis=1), color="black", lw=2.5, label="Mean")

    ax.axvline(0, color="k", linestyle="--")
    ax.set_title(f"Peri-event dF/F — {example_key} | Behavior: {beh}")
    ax.set_xlabel("Time relative to event start (s)")
    ax.set_ylabel("ΔF/F (%)")
    ax.grid(True)
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()
    if len(event_starts_idx) == 0:
        print(f"No events found for {beh}")
        continue

    # Convert pre/post window to samples
    fps = 1 / np.mean(np.diff(time_vec))  # effective sampling rate
    pre_n = int(pre_s * fps)
    post_n = int(post_s * fps)

    # Collect peri-event segments
    segments = []
    for idx in event_starts_idx:
        if idx >= pre_n and (idx + post_n) < len(signal_vec):
            seg = signal_vec[idx-pre_n : idx+post_n].copy()
            seg = seg - np.mean(seg[:pre_n])  # baseline correction
            segments.append(seg)

    if len(segments) == 0:
        print(f"No valid peri-event segments for {beh}")
        continue

    ERF = np.column_stack(segments)
    t_window = np.linspace(-pre_s, post_s, pre_n + post_n)

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(8,4))
    # Individual traces
    colors = plt.cm.tab10(np.linspace(0,1,ERF.shape[1]))
    for i, col in enumerate(colors):
        ax.plot(t_window, ERF[:,i], color=col, alpha=0.3, lw=1)
    # Mean trace
    ax.plot(t_window, ERF.mean(axis=1), color="black", lw=2.5, label="Mean")

    ax.axvline(0, color="k", linestyle="--")
    ax.set_title(f"Peri-event dF/F — {example_key} | Behavior: {beh}")
    ax.set_xlabel("Time relative to event start (s)")
    ax.set_ylabel("ΔF/F (%)")
    ax.grid(True)
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np
import pandas as pd

# Pick the trial
example_key = list(combined_trials.keys())[1]
df = combined_trials[example_key]

behaviors = df["Behavior"].values
unique_beh = np.unique(behaviors)

# Count number of instances (rising edges into each behavior)
event_counts = {}

for beh in unique_beh:
    mask = (behaviors == beh)
    mask_int = mask.astype(int)
    diffs = np.diff(np.r_[0, mask_int])  # prepend 0 for first element
    event_starts_idx = np.where(diffs == 1)[0]  # transitions into behavior
    event_counts[beh] = len(event_starts_idx)

# Convert to DataFrame for display
event_counts_df = pd.DataFrame.from_dict(event_counts, orient="index", columns=["Num_Instances"])
event_counts_df.index.name = "Behavior"
event_counts_df


# event plots split by condition



# Mouse grand mean plots

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# # ---------- PARAMETERS ----------
# pre_s = 2.0
# post_s = 4.0

# # ---------- GLOBAL FPS ----------
# all_dts = []
# for trial_key, trial_data in combined_trials.items():
#     time_vec = trial_data["Time_video"].values
#     all_dts.extend(np.diff(time_vec))

# global_fps = 1 / np.median(all_dts)  # robust to noise

# pre_n = int(pre_s * global_fps)
# post_n = int(post_s * global_fps)
# expected_len = pre_n + post_n

# t_window = np.linspace(-pre_s, post_s, expected_len)

# # ---------- COLLECT BEHAVIORS ----------
# all_behaviors = set()
# for trial_key, trial_data in combined_trials.items():
#     if fp_traces[trial_key].get("condition") != "Social":
#         continue
#     behs = np.unique(trial_data["Behavior"].values)
#     all_behaviors.update([b for b in behs if b != "Other"])

# all_behaviors = sorted(list(all_behaviors))

# # ---------- MAIN LOOP ----------
# for beh in all_behaviors:
#     mouse_segments = []
#     mouse_ids = []

#     for trial_key, trial_data in combined_trials.items():
#         fp_info = fp_traces[trial_key]
#         if fp_info.get("condition") != "Social":
#             continue

#         df = trial_data.copy()
#         time_vec = df["Time_video"].values
#         signal_vec = df["dFF"].values
#         behaviors = df["Behavior"].values

#         if beh not in behaviors:
#             continue

#         # Find event starts
#         mask = behaviors == beh
#         diffs = np.diff(np.r_[0, mask.astype(int)])
#         event_starts_idx = np.where(diffs == 1)[0]

#         if len(event_starts_idx) == 0:
#             continue

#         segments = []
#         for idx in event_starts_idx:
#             if idx >= pre_n and (idx + post_n) < len(signal_vec):
#                 seg = signal_vec[idx-pre_n : idx+post_n].copy()

#                 # ✅ enforce fixed length
#                 if len(seg) != expected_len:
#                     continue

#                 seg = seg - np.mean(seg[:pre_n])
#                 segments.append(seg)

#         if len(segments) == 0:
#             continue

#         mouse_mean = np.mean(np.column_stack(segments), axis=1)

#         # ✅ final safety check
#         if len(mouse_mean) != expected_len:
#             continue

#         mouse_segments.append(mouse_mean)
#         mouse_ids.append(fp_info.get("mouse_id", trial_key))

#     if len(mouse_segments) == 0:
#         print(f"No valid events for behavior {beh}")
#         continue

#     # ---------- PLOT ----------
#     fig, ax = plt.subplots(figsize=(8, 4))

#     for i, seg in enumerate(mouse_segments):
#         color = plt.cm.tab20(i % 20)
#         ax.plot(t_window, seg, color=color, alpha=0.4, lw=1.5,
#                 label=f"Mouse {mouse_ids[i]}")

#     # Grand mean
#     grand_mean = np.mean(np.column_stack(mouse_segments), axis=1)
#     ax.plot(t_window, grand_mean, color="black", lw=2.5, label="Grand mean")

#     ax.axvline(0, color="k", linestyle="--")
#     ax.set_title(f"Peri-event dF/F — Behavior: {beh} (Social trials)")
#     ax.set_xlabel("Time relative to event start (s)")
#     ax.set_ylabel("ΔF/F (%)")
#     ax.grid(True)
#     ax.legend(frameon=False)

#     plt.tight_layout()
#     plt.show()

# Novelty

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# # ---------- PARAMETERS ----------
# pre_s = 2.0
# post_s = 4.0

# # Collect all behaviors across Social trials
# all_behaviors = set()
# for trial_key, trial_data in combined_trials.items():
#     if fp_traces[trial_key].get("condition") != "Novel":
#         continue
#     behs = np.unique(trial_data["Behavior"].values)
#     all_behaviors.update([b for b in behs if b != "Other"])  # skip "Other"

# all_behaviors = sorted(list(all_behaviors))

# # Loop over behaviors to pool across mice
# for beh in all_behaviors:
#     mouse_segments = []
#     mouse_ids = []

#     for trial_key, trial_data in combined_trials.items():
#         fp_info = fp_traces[trial_key]
#         if fp_info.get("condition") != "Novel":
#             continue

#         df = trial_data.copy()
#         time_vec = df["Time_video"].values
#         signal_vec = df["dFF"].values
#         behaviors = df["Behavior"].values

#         if beh not in behaviors:
#             continue

#         # Find event starts
#         mask = behaviors == beh
#         mask_int = mask.astype(int)
#         diffs = np.diff(np.r_[0, mask_int])
#         event_starts_idx = np.where(diffs == 1)[0]

#         if len(event_starts_idx) == 0:
#             continue

#         fps = 1 / np.mean(np.diff(time_vec))
#         pre_n = int(pre_s * fps)
#         post_n = int(post_s * fps)

#         segments = []
#         for idx in event_starts_idx:
#             if idx >= pre_n and (idx + post_n) < len(signal_vec):
#                 seg = signal_vec[idx-pre_n : idx+post_n].copy()
#                 seg = seg - np.mean(seg[:pre_n])
#                 segments.append(seg)

#         if len(segments) == 0:
#             continue

#         # Average across events for this mouse
#         mouse_mean = np.mean(np.column_stack(segments), axis=1)
#         mouse_segments.append(mouse_mean)
#         mouse_ids.append(fp_info.get("mouse_id", trial_key))

#     if len(mouse_segments) == 0:
#         print(f"No valid events for behavior {beh}")
#         continue

#     # --- Pooled plot ---
#     t_window = np.linspace(-pre_s, post_s, pre_n + post_n)
#     fig, ax = plt.subplots(figsize=(8,4))

#     # Plot each mouse
#     for i, seg in enumerate(mouse_segments):
#         color = plt.cm.tab20(i % 20)  # up to 20 distinct colors
#         ax.plot(t_window, seg, color=color, alpha=0.4, lw=1.5, label=f"Mouse {mouse_ids[i]}")

#     # Plot grand mean
#     grand_mean = np.mean(np.column_stack(mouse_segments), axis=1)
#     ax.plot(t_window, grand_mean, color="black", lw=2.5, label="Grand mean")

#     ax.axvline(0, color="k", linestyle="--")
#     ax.set_title(f"Peri-event dF/F — Behavior: {beh} (Social trials)")
#     ax.set_xlabel("Time relative to event start (s)")
#     ax.set_ylabel("ΔF/F (%)")
#     ax.grid(True)
#     ax.legend(frameon=False)
#     plt.tight_layout()
#     plt.show()


In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# # ---------- PARAMETERS ----------
# pre_s = 2.0
# post_s = 4.0

# # ---------- GLOBAL FPS ----------
# all_dts = []
# for trial_key, trial_data in combined_trials.items():
#     time_vec = trial_data["Time_video"].values
#     all_dts.extend(np.diff(time_vec))

# global_fps = 1 / np.median(all_dts)
# pre_n = int(pre_s * global_fps)
# post_n = int(post_s * global_fps)
# expected_len = pre_n + post_n
# t_window = np.linspace(-pre_s, post_s, expected_len)

# # ---------- METADATA ----------
# all_behaviors = set()
# all_weeks = set()
# all_genotypes = set()
# all_mouse_ids = set()
# for trial_key, trial_data in combined_trials.items():
#     fp_info = fp_traces[trial_key]
#     if fp_info.get("condition") != "Social":
#         continue
#     behs = np.unique(trial_data["Behavior"].values)
#     all_behaviors.update([b for b in behs if b != "Other"])
#     all_weeks.add(fp_info.get("week"))
#     all_genotypes.add(fp_info.get("genotype"))
#     all_mouse_ids.add(fp_info.get("mouse_id", trial_key))

# all_behaviors = sorted(all_behaviors)
# all_weeks = sorted(all_weeks)
# all_genotypes = sorted(all_genotypes)
# all_mouse_ids = sorted(all_mouse_ids)

# # Assign a color per mouse
# mouse_colors = {mid: plt.cm.tab20(i % 20) for i, mid in enumerate(all_mouse_ids)}

# # =====================================================
# #      MAIN LOOP: BEHAVIOR × GENOTYPE × WEEK
# # =====================================================
# for beh in all_behaviors:
#     fig, axes = plt.subplots(
#         len(all_weeks),
#         len(all_genotypes),
#         figsize=(6 * len(all_genotypes), 4 * len(all_weeks)),
#         sharey=True,
#         sharex=True
#     )

#     # ensure 2D axes
#     if len(all_weeks) == 1:
#         axes = np.array([axes])
#     if len(all_genotypes) == 1:
#         axes = axes[:, np.newaxis]

#     for r, week in enumerate(all_weeks):
#         for c, genotype in enumerate(all_genotypes):
#             ax = axes[r, c]

#             # ---------- GATHER DATA ----------
#             instance_segments = []
#             instance_mouse_ids = []

#             for trial_key, trial_data in combined_trials.items():
#                 fp_info = fp_traces[trial_key]
#                 if fp_info.get("condition") != "Social":
#                     continue
#                 if fp_info.get("week") != week:
#                     continue
#                 if fp_info.get("genotype") != genotype:
#                     continue

#                 df = trial_data.copy()
#                 time_vec = df["Time_video"].values
#                 signal_vec = df["dFF"].values
#                 behaviors = df["Behavior"].values
#                 mouse_id = fp_info.get("mouse_id", trial_key)

#                 if beh not in behaviors:
#                     continue

#                 # Event starts
#                 starts = np.where(np.diff(np.r_[0, (behaviors == beh).astype(int)]) == 1)[0]
#                 if len(starts) == 0:
#                     continue

#                 for idx in starts:
#                     if idx >= pre_n and (idx + post_n) < len(signal_vec):
#                         seg = signal_vec[idx-pre_n:idx+post_n].copy()
#                         if len(seg) != expected_len:
#                             continue
#                         seg -= np.mean(seg[:pre_n])
#                         instance_segments.append(seg)
#                         instance_mouse_ids.append(mouse_id)

#             if len(instance_segments) == 0:
#                 ax.set_title(f"{week} | {genotype}\n(no events)")
#                 ax.grid(True)
#                 continue

#             instance_segments = np.vstack(instance_segments)

#             # ---------- PLOTTING ----------
#             for seg, mid in zip(instance_segments, instance_mouse_ids):
#                 ax.plot(t_window, seg, color=mouse_colors[mid], alpha=0.5, lw=1.5)

#             # Mean across all instances
#             grand_mean = np.mean(instance_segments, axis=0)
#             ax.plot(t_window, grand_mean, color="black", lw=2.5)

#             # SEM across all instances
#             sem = np.std(instance_segments, axis=0) / np.sqrt(instance_segments.shape[0])
#             ax.fill_between(t_window, grand_mean - sem, grand_mean + sem, color="black", alpha=0.2)

#             ax.axvline(0, color='k', linestyle='--')
#             ax.set_title(f"{week} | {genotype}\n(n_instances={instance_segments.shape[0]})")
#             ax.grid(True)

#             if r == len(all_weeks) - 1:
#                 ax.set_xlabel("Time (s)")
#             if c == 0:
#                 ax.set_ylabel("ΔF/F (%)")

#     plt.suptitle(f"Peri-event dF/F — Behavior: {beh} (Social Trial)", fontsize=18)
#     plt.tight_layout()
#     plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------- PARAMETERS ----------
pre_s = 2.0
post_s = 4.0

# ---------- GLOBAL FPS ----------
all_dts = []
for trial_key, trial_data in combined_trials.items():
    time_vec = trial_data["Time_video"].values
    all_dts.extend(np.diff(time_vec))

global_fps = 1 / np.median(all_dts)
pre_n = int(pre_s * global_fps)
post_n = int(post_s * global_fps)
expected_len = pre_n + post_n
t_window = np.linspace(-pre_s, post_s, expected_len)

# ---------- METADATA ----------
all_behaviors = set()
all_weeks = set()
all_genotypes = set()
for trial_key, trial_data in combined_trials.items():
    fp_info = fp_traces[trial_key]
    if fp_info.get("condition") != "Social":
        continue
    behs = np.unique(trial_data["Behavior"].values)
    all_behaviors.update([b for b in behs if b != "Other"])
    all_weeks.add(fp_info.get("week"))
    all_genotypes.add(fp_info.get("genotype"))

all_behaviors = sorted(all_behaviors)
all_weeks = sorted(all_weeks)
all_genotypes = sorted(all_genotypes)

# =====================================================
#      MAIN LOOP: BEHAVIOR × GENOTYPE × WEEK
# =====================================================
for beh in all_behaviors:
    fig, axes = plt.subplots(
        len(all_weeks),
        len(all_genotypes),
        figsize=(6 * len(all_genotypes), 4 * len(all_weeks)),
        sharey=True,
        sharex=True
    )

    # ensure 2D axes
    if len(all_weeks) == 1:
        axes = np.array([axes])
    if len(all_genotypes) == 1:
        axes = axes[:, np.newaxis]

    for r, week in enumerate(all_weeks):
        for c, genotype in enumerate(all_genotypes):
            ax = axes[r, c]

            mouse_segments = []
            mouse_ids = []

            # ---------- GATHER DATA ----------
            for trial_key, trial_data in combined_trials.items():
                fp_info = fp_traces[trial_key]
                if fp_info.get("condition") != "Social":
                    continue
                if fp_info.get("week") != week:
                    continue
                if fp_info.get("genotype") != genotype:
                    continue

                df = trial_data.copy()
                time_vec = df["Time_video"].values
                signal_vec = df["dFF"].values
                behaviors = df["Behavior"].values

                if beh not in behaviors:
                    continue

                # Event starts
                starts = np.where(np.diff(np.r_[0, (behaviors == beh).astype(int)]) == 1)[0]
                if len(starts) == 0:
                    continue

                segments = []
                for idx in starts:
                    if idx >= pre_n and (idx + post_n) < len(signal_vec):
                        seg = signal_vec[idx-pre_n:idx+post_n].copy()
                        if len(seg) != expected_len:
                            continue
                        seg -= np.mean(seg[:pre_n])
                        segments.append(seg)

                if len(segments) == 0:
                    continue

                mouse_mean = np.mean(np.column_stack(segments), axis=1)
                if len(mouse_mean) != expected_len:
                    continue

                mouse_segments.append(mouse_mean)
                mouse_ids.append(fp_info.get("mouse_id", trial_key))

            if len(mouse_segments) == 0:
                ax.set_title(f"{week} | {genotype}\n(no events)")
                ax.grid(True)
                continue

            mouse_segments = np.vstack(mouse_segments)

            # ---------- PLOTTING ----------
            for i, seg in enumerate(mouse_segments):
                ax.plot(t_window, seg, color=plt.cm.tab20(i % 20), alpha=0.4, lw=1.5)

            # mean trace
            grand_mean = np.mean(mouse_segments, axis=0)
            ax.plot(t_window, grand_mean, color="black", lw=2.5)

            # ---------- SEM SHADING ----------
            sem = np.std(mouse_segments, axis=0) / np.sqrt(mouse_segments.shape[0])
            ax.fill_between(t_window, grand_mean - sem, grand_mean + sem,
                            color="black", alpha=0.2)

            ax.axvline(0, color='k', linestyle='--')
            ax.set_title(f"{week} | {genotype}\n(n={mouse_segments.shape[0]})")
            ax.grid(True)

            if r == len(all_weeks) - 1:
                ax.set_xlabel("Time (s)")
            if c == 0:
                ax.set_ylabel("ΔF/F (%)")

    plt.suptitle(f"Peri-event dF/F — Behavior: {beh} (Social Trial)", fontsize=18)
    plt.tight_layout()
    plt.show()

# Novelty

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------- PARAMETERS ----------
pre_s = 2.0
post_s = 4.0

# ---------- GLOBAL FPS ----------
all_dts = []
for trial_key, trial_data in combined_trials.items():
    time_vec = trial_data["Time_video"].values
    all_dts.extend(np.diff(time_vec))

global_fps = 1 / np.median(all_dts)
pre_n = int(pre_s * global_fps)
post_n = int(post_s * global_fps)
expected_len = pre_n + post_n
t_window = np.linspace(-pre_s, post_s, expected_len)

# ---------- METADATA ----------
all_behaviors = set()
all_weeks = set()
all_genotypes = set()
for trial_key, trial_data in combined_trials.items():
    fp_info = fp_traces[trial_key]
    if fp_info.get("condition") != "Novel":
        continue
    behs = np.unique(trial_data["Behavior"].values)
    all_behaviors.update([b for b in behs if b != "Other"])
    all_weeks.add(fp_info.get("week"))
    all_genotypes.add(fp_info.get("genotype"))

all_behaviors = sorted(all_behaviors)
all_weeks = sorted(all_weeks)
all_genotypes = sorted(all_genotypes)

# =====================================================
#      MAIN LOOP: BEHAVIOR × GENOTYPE × WEEK
# =====================================================
for beh in all_behaviors:
    fig, axes = plt.subplots(
        len(all_weeks),
        len(all_genotypes),
        figsize=(6 * len(all_genotypes), 4 * len(all_weeks)),
        sharey=True,
        sharex=True
    )

    # ensure 2D axes
    if len(all_weeks) == 1:
        axes = np.array([axes])
    if len(all_genotypes) == 1:
        axes = axes[:, np.newaxis]

    for r, week in enumerate(all_weeks):
        for c, genotype in enumerate(all_genotypes):
            ax = axes[r, c]

            mouse_segments = []
            mouse_ids = []

            # ---------- GATHER DATA ----------
            for trial_key, trial_data in combined_trials.items():
                fp_info = fp_traces[trial_key]
                if fp_info.get("condition") != "Novel":
                    continue
                if fp_info.get("week") != week:
                    continue
                if fp_info.get("genotype") != genotype:
                    continue

                df = trial_data.copy()
                time_vec = df["Time_video"].values
                signal_vec = df["dFF"].values
                behaviors = df["Behavior"].values

                if beh not in behaviors:
                    continue

                # Event starts
                starts = np.where(np.diff(np.r_[0, (behaviors == beh).astype(int)]) == 1)[0]
                if len(starts) == 0:
                    continue

                segments = []
                for idx in starts:
                    if idx >= pre_n and (idx + post_n) < len(signal_vec):
                        seg = signal_vec[idx-pre_n:idx+post_n].copy()
                        if len(seg) != expected_len:
                            continue
                        seg -= np.mean(seg[:pre_n])
                        segments.append(seg)

                if len(segments) == 0:
                    continue

                mouse_mean = np.mean(np.column_stack(segments), axis=1)
                if len(mouse_mean) != expected_len:
                    continue

                mouse_segments.append(mouse_mean)
                mouse_ids.append(fp_info.get("mouse_id", trial_key))

            if len(mouse_segments) == 0:
                ax.set_title(f"{week} | {genotype}\n(no events)")
                ax.grid(True)
                continue

            mouse_segments = np.vstack(mouse_segments)

            # ---------- PLOTTING ----------
            for i, seg in enumerate(mouse_segments):
                ax.plot(t_window, seg, color=plt.cm.tab20(i % 20), alpha=0.4, lw=1.5)

            # mean trace
            grand_mean = np.mean(mouse_segments, axis=0)
            ax.plot(t_window, grand_mean, color="black", lw=2.5)

            # ---------- SEM SHADING ----------
            sem = np.std(mouse_segments, axis=0) / np.sqrt(mouse_segments.shape[0])
            ax.fill_between(t_window, grand_mean - sem, grand_mean + sem,
                            color="black", alpha=0.2)

            ax.axvline(0, color='k', linestyle='--')
            ax.set_title(f"{week} | {genotype}\n(n={mouse_segments.shape[0]})")
            ax.grid(True)

            if r == len(all_weeks) - 1:
                ax.set_xlabel("Time (s)")
            if c == 0:
                ax.set_ylabel("ΔF/F (%)")

    plt.suptitle(f"Peri-event dF/F — Behavior: {beh} (Novel Trial)", fontsize=18)
    plt.tight_layout()
    plt.show()

# Genotype plots

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# # ---------- PARAMETERS ----------
# pre_s = 2.0
# post_s = 4.0

# # ---------- COLLECT METADATA ----------
# all_behaviors = set()
# all_weeks = set()
# all_genotypes = set()

# for trial_key, trial_data in combined_trials.items():
#     fp_info = fp_traces[trial_key]

#     if fp_info.get("condition") != "Social":
#         continue

#     behs = np.unique(trial_data["Behavior"].values)
#     all_behaviors.update([b for b in behs if b != "Other"])

#     all_weeks.add(fp_info.get("week"))
#     all_genotypes.add(fp_info.get("genotype"))

# all_behaviors = sorted(all_behaviors)
# all_weeks = sorted(all_weeks)
# all_genotypes = sorted(all_genotypes)


# # =====================================================
# #      MAIN LOOP: BEHAVIOR × GENOTYPE × WEEK
# # =====================================================
# for beh in all_behaviors:

#     fig, axes = plt.subplots(
#         len(all_weeks),
#         len(all_genotypes),
#         figsize=(6 * len(all_genotypes), 4 * len(all_weeks)),
#         sharey=True,
#         sharex=True
#     )

#     # ensure 2D axis array
#     if len(all_weeks) == 1:
#         axes = [axes]
#     if len(all_genotypes) == 1:
#         axes = [[axes[r]] for r in range(len(axes))]

#     for r, week in enumerate(all_weeks):
#         for c, genotype in enumerate(all_genotypes):
            
#             ax = axes[r][c]

#             mouse_segments = []
#             mouse_ids = []
#             mouse_lengths = []

#             # ----------------------------------------
#             #          Gather data for cell
#             # ----------------------------------------
#             for trial_key, trial_data in combined_trials.items():
#                 fp_info = fp_traces[trial_key]

#                 if fp_info.get("condition") != "Novel":
#                     continue
#                 if fp_info.get("week") != week:
#                     continue
#                 if fp_info.get("genotype") != genotype:
#                     continue

#                 df = trial_data.copy()
#                 time_vec = df["Time_video"].values
#                 signal_vec = df["dFF"].values
#                 behaviors = df["Behavior"].values

#                 if beh not in behaviors:
#                     continue

#                 # ---------- EVENT STARTS ----------
#                 starts = np.where(np.diff(np.r_[0, (behaviors == beh).astype(int)]) == 1)[0]
#                 if len(starts) == 0:
#                     continue

#                 fps = 1.0 / np.mean(np.diff(time_vec))
#                 pre_n = int(pre_s * fps)
#                 post_n = int(post_s * fps)

#                 event_segments = []

#                 for idx in starts:
#                     start_idx = idx - pre_n
#                     stop_idx = idx + post_n

#                     seg_raw = signal_vec[max(0, start_idx):min(stop_idx, len(signal_vec))].copy()

#                     # baseline
#                     if len(seg_raw) > 0:
#                         pre_end = min(pre_n, len(seg_raw))
#                         seg_raw -= np.mean(seg_raw[:pre_end])

#                     event_segments.append(seg_raw)

#                 if len(event_segments) == 0:
#                     continue

#                 # pad event segments to same length within this mouse
#                 max_len_mouse = max(len(s) for s in event_segments)
#                 padded_events = []
#                 for s in event_segments:
#                     padded = np.full(max_len_mouse, np.nan)
#                     padded[:len(s)] = s
#                     padded_events.append(padded)

#                 mouse_mean = np.nanmean(np.vstack(padded_events), axis=0)
#                 mouse_segments.append(mouse_mean)
#                 mouse_ids.append(fp_info.get("mouse_id", trial_key))
#                 mouse_lengths.append(len(mouse_mean))

#             # =================================================
#             #     NOTHING FOR THIS WEEK × GENOTYPE CELL
#             # =================================================
#             if len(mouse_segments) == 0:
#                 ax.set_title(f"{week} | {genotype}\n(no events)")
#                 ax.grid(True)
#                 continue

#             # =================================================
#             #     DETERMINE GLOBAL TARGET LENGTH FOR CELL
#             # =================================================
#             target_len = max(mouse_lengths)

#             # pad all mice to this target_len
#             padded_mouse_segments = []
#             for seg in mouse_segments:
#                 padded = np.full(target_len, np.nan)
#                 padded[:len(seg)] = seg
#                 padded_mouse_segments.append(padded)

#             padded_mouse_segments = np.vstack(padded_mouse_segments)

#             # make time vector for this cell
#             t_window = np.linspace(-pre_s, post_s, target_len)

#             # =================================================
#             #                 PLOTTING
#             # =================================================
#             for i, seg in enumerate(padded_mouse_segments):
#                 ax.plot(t_window, seg, color=plt.cm.tab20(i % 20),
#                         alpha=0.4, lw=1.5)

#             # mean trace
#             grand_mean = np.nanmean(padded_mouse_segments, axis=0)
#             ax.plot(t_window, grand_mean, color="black", lw=2.5)

#             ax.axvline(0, color='k', linestyle='--')
#             ax.set_title(f"{week} | {genotype}")
#             ax.grid(True)

#             if r == len(all_weeks) - 1:
#                 ax.set_xlabel("Time (s)")
#             if c == 0:
#                 ax.set_ylabel("ΔF/F (%)")

#     plt.suptitle(f"Peri-event dF/F — Behavior: {beh} (Social)", fontsize=18)
#     plt.tight_layout()
#     plt.show()


In [ ]:
import numpy as np
import pandas as pd

# -------------------------------------------------------------
# BUILD mean_summary_df from combined_trials + fp_traces
# -------------------------------------------------------------

mean_rows = []

for trial_key, df in combined_trials.items():

    fp_info = fp_traces[trial_key]

    mouse_id  = fp_info.get("mouse_id")
    genotype  = fp_info.get("genotype")
    condition = fp_info.get("condition")
    week      = fp_info.get("week")

    # behavior labels + signal
    beh = df["Behavior"].values
    dff = df["dFF"].values

    # list of behaviors in the trial
    unique_behaviors = np.unique(beh)

    for b in unique_behaviors:

        mask = beh == b
        if mask.sum() == 0:
            continue

        mean_dff = np.nanmean(dff[mask])
        npoints  = mask.sum()

        mean_rows.append({
            "mouse_id":  mouse_id,
            "genotype":  genotype,
            "condition": condition,
            "week":      week,
            "behavior":  b,
            "mean_dFF":  mean_dff,
            "n_points":  npoints,
            "trial_key": trial_key
        })

mean_summary_df = pd.DataFrame(mean_rows)

print("Created mean_summary_df with shape:", mean_summary_df.shape)
mean_summary_df.head()


In [ ]:
print("combined_trials exists:", 'combined_trials' in globals())
print("fp_traces exists:", 'fp_traces' in globals())


In [ ]:
import numpy as np
import pandas as pd

# Safety checks
if 'combined_trials' not in globals():
    raise NameError("combined_trials is not defined. Run the preprocessing cells first.")

if 'fp_traces' not in globals():
    raise NameError("fp_traces is not defined. Run the preprocessing cells first.")

mean_summary = []

for trial_key, trial_data in combined_trials.items():

    if "dFF" not in trial_data.columns:
        print(f"[WARN] Skipping {trial_key}: no dFF column")
        continue

    if "Behavior" not in trial_data.columns:
        print(f"[WARN] Skipping {trial_key}: no Behavior column")
        continue

    meta = fp_traces.get(trial_key, {})
    mouse_id  = meta.get("mouse_id", None)
    genotype  = meta.get("genotype", None)
    condition = meta.get("condition", None)
    week      = meta.get("week", None)

    df = trial_data.copy()

    dff = df["dFF"].values
    beh = df["Behavior"].values

    behaviors = np.unique(beh)

    for b in behaviors:
        mask = (beh == b)
        if mask.sum() == 0:
            continue

        mean_dff = np.nanmean(dff[mask])

        mean_summary.append({
            "mouse_id": mouse_id,
            "genotype": genotype,
            "condition": condition,
            "week": week,
            "behavior": b,
            "mean_dFF": mean_dff,
            "n_points": mask.sum()
        })

mean_summary_df = pd.DataFrame(mean_summary)
print("Summary table created. Shape:", mean_summary_df.shape)
print(mean_summary_df.head())


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def plot_behavior_facet_genotype(mean_df, condition_filter=None):
    """
    For each behavior (excluding 'Other'):
        • One figure
        • x-axis = week (Preinduction, W1, W2, W3)
        • facet column = genotype (NE left, WT right)
        • bars = mean ± SEM
        • points = each mouse
    """

    df = mean_df.copy()

    # Filter by condition (optional)
    if condition_filter is not None:
        df = df[df["condition"] == condition_filter]

    # Remove "Other"
    df = df[df["behavior"] != "Other"]

    if df.empty:
        print(f"[WARN] No data available after filtering condition='{condition_filter}' and excluding 'Other'")
        return

    # Ensure ordering
    week_order = ["Preinduction", "W1", "W2", "W3"]
    df["week"] = pd.Categorical(df["week"], categories=week_order, ordered=True)

    genotype_order = ["NE", "WT"]
    df["genotype"] = pd.Categorical(df["genotype"], categories=genotype_order, ordered=True)

    # Behavior list (excluding Other already)
    behaviors = sorted(df["behavior"].unique())

    for beh in behaviors:

        sub = df[df["behavior"] == beh]

        if sub.empty:
            continue

        # Facet by genotype (NE left, WT right)
        g = sns.FacetGrid(
            sub,
            col="genotype",
            col_order=genotype_order,
            height=5,
            aspect=1.1,
            sharey=True
        )

        # Mean ± SEM bars
        g.map_dataframe(
            sns.barplot,
            x="week",
            y="mean_dFF",
            order=week_order,
            palette="Set2",
            errorbar="se",
            width=0.7,
            alpha=0.85
        )

        # Individual datapoints
        g.map_dataframe(
            sns.stripplot,
            x="week",
            y="mean_dFF",
            order=week_order,
            color="black",
            alpha=0.7,
            jitter=True,
            size=5
        )

        g.set_axis_labels("Week", "Mean ΔF/F (%)")
        g.set_titles("{col_name}")  # NE / WT as titles
        g.fig.suptitle(
            f"Behavior: {beh} — ΔF/F across weeks",
            fontsize=16,
            y=1.05
        )

        # Style
        for ax in g.axes.flatten():
            ax.grid(axis="y", alpha=0.3)
            ax.set_xlabel("")

        plt.tight_layout()
        plt.show()

# import seaborn as sns
# import matplotlib.pyplot as plt
# import pandas as pd
# import numpy as np

# def plot_behavior_facet_genotype(mean_df, condition_filter=None):
#     """
#     For each behavior (excluding 'Other'):
#         • One figure
#         • x-axis = week (Preinduction, W1, W2, W3)
#         • facet column = genotype (NE left, WT right)
#         • bars = mean ± SEM
#         • points = each mouse, colored by mouse_id
#     """

#     df = mean_df.copy()

#     # Filter by condition (optional)
#     if condition_filter is not None:
#         df = df[df["condition"] == condition_filter]

#     # Remove "Other"
#     df = df[df["behavior"] != "Other"]

#     if df.empty:
#         print(f"[WARN] No data available after filtering condition='{condition_filter}' and excluding 'Other'")
#         return

#     # Ensure ordering
#     week_order = ["Preinduction", "W1", "W2", "W3"]
#     df["week"] = pd.Categorical(df["week"], categories=week_order, ordered=True)

#     genotype_order = ["NE", "WT"]
#     df["genotype"] = pd.Categorical(df["genotype"], categories=genotype_order, ordered=True)

#     # Behavior list
#     behaviors = sorted(df["behavior"].unique())

#     for beh in behaviors:

#         sub = df[df["behavior"] == beh]

#         if sub.empty:
#             continue

#         # Get unique mouse IDs for consistent palette
#         mouse_ids = sub["mouse_id"].unique()
#         palette = sns.color_palette("tab20", n_colors=len(mouse_ids))
#         mouse_palette = dict(zip(mouse_ids, palette))

#         # Facet by genotype (NE left, WT right)
#         g = sns.FacetGrid(
#             sub,
#             col="genotype",
#             col_order=genotype_order,
#             height=5,
#             aspect=1.1,
#             sharey=True
#         )

#         # Mean ± SEM bars
#         g.map_dataframe(
#             sns.barplot,
#             x="week",
#             y="mean_dFF",
#             order=week_order,
#             palette="Set2",
#             errorbar="se",
#             width=0.7,
#             alpha=0.85
#         )

#         # Individual datapoints colored by mouse_id
#         g.map_dataframe(
#             sns.stripplot,
#             x="week",
#             y="mean_dFF",
#             hue="mouse_id",
#             palette=mouse_palette,
#             dodge=True,
#             alpha=0.8,
#             size=5,
#             jitter=True
#         )

#         # Remove redundant legend per axis
#         for ax in g.axes.flatten():
#             ax.legend_.remove()

#         # Add a single legend for the entire figure
#         handles, labels = ax.get_legend_handles_labels()
#         g.fig.legend(handles, labels, title="Mouse ID", bbox_to_anchor=(0.92, 0.85), frameon=False)

#         g.set_axis_labels("Week", "Mean ΔF/F (%)")
#         g.set_titles("{col_name}")  # NE / WT as titles
#         g.fig.suptitle(
#             f"Behavior: {beh} — ΔF/F across weeks",
#             fontsize=16,
#             y=1.05
#         )

#         # Style
#         for ax in g.axes.flatten():
#             ax.grid(axis="y", alpha=0.3)
#             ax.set_xlabel("")

#         plt.tight_layout()
#         plt.show()

In [ ]:
plot_behavior_facet_genotype(mean_summary_df, condition_filter="Social")


In [ ]:
plot_behavior_facet_genotype(mean_summary_df, condition_filter="Novel")


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --------------------------
# PARAMETERS
# --------------------------
pre_s = 2.0        # seconds before event
post_s = 4.0       # seconds after event
post2s_s = 2.0     # seconds to include for post-event mean (0–2 s)

# --------------------------
# COMPUTE GLOBAL FPS
# --------------------------
all_dts = []
for trial_key, trial_data in combined_trials.items():
    if "Time_video" not in trial_data.columns:
        continue
    time_vec = trial_data["Time_video"].values
    all_dts.extend(np.diff(time_vec))

global_fps = 1 / np.median(all_dts)
pre_n = int(pre_s * global_fps)
post_n = int(post_s * global_fps)
expected_len = pre_n + post_n
t_window = np.linspace(-pre_s, post_s, expected_len)
post2s_n = int(post2s_s * global_fps)  # number of points to average 0–2 s

# --------------------------
# CREATE SUMMARY TABLE FOR 0–2 s POST EVENT
# --------------------------
summary_post2s = []

for trial_key, trial_data in combined_trials.items():
    fp_info = fp_traces.get(trial_key, {})
    condition = fp_info.get("condition")
    week = fp_info.get("week")
    genotype = fp_info.get("genotype")
    mouse_id = fp_info.get("mouse_id", trial_key)

    if "dFF" not in trial_data.columns or "Behavior" not in trial_data.columns:
        continue

    signal = trial_data["dFF"].values
    behaviors = trial_data["Behavior"].values
    time_vec = trial_data["Time_video"].values

    unique_behaviors = [b for b in np.unique(behaviors) if b != "Other"]

    for beh in unique_behaviors:
        # Find event starts
        starts = np.where(np.diff(np.r_[0, (behaviors == beh).astype(int)]) == 1)[0]
        if len(starts) == 0:
            continue

        segments = []
        for idx in starts:
            if idx >= pre_n and (idx + post_n) < len(signal):
                seg = signal[idx-pre_n:idx+post_n].copy()
                seg -= np.mean(seg[:pre_n])  # baseline correction
                segments.append(seg)

        if len(segments) == 0:
            continue

        mouse_mean_seg = np.mean(np.column_stack(segments), axis=1)
        post2s_mean = np.mean(mouse_mean_seg[pre_n:pre_n+post2s_n])  # 0–2 s post-event

        summary_post2s.append({
            "mouse_id": mouse_id,
            "genotype": genotype,
            "condition": condition,
            "week": week,
            "behavior": beh,
            "mean_dFF": post2s_mean,
            "n_events": len(segments)
        })

mean_summary_post2s_df = pd.DataFrame(summary_post2s)
print("0–2 s post-event summary created. Shape:", mean_summary_post2s_df.shape)
print(mean_summary_post2s_df.head())

# --------------------------
# PLOTTING FUNCTION (FACET BY GENOTYPE)
# --------------------------
def plot_behavior_facet_genotype_post2s(summary_df, condition_filter=None):
    df = summary_df.copy()

    if condition_filter is not None:
        df = df[df["condition"] == condition_filter]

    df = df[df["behavior"] != "Other"]

    if df.empty:
        print(f"[WARN] No data available after filtering.")
        return

    week_order = ["Preinduction", "W1", "W2", "W3"]
    df["week"] = pd.Categorical(df["week"], categories=week_order, ordered=True)

    genotype_order = ["NE", "WT"]
    df["genotype"] = pd.Categorical(df["genotype"], categories=genotype_order, ordered=True)

    behaviors = sorted(df["behavior"].unique())

    for beh in behaviors:
        sub = df[df["behavior"] == beh]
        if sub.empty:
            continue

        mouse_ids = sub["mouse_id"].unique()
        palette = sns.color_palette("tab20", n_colors=len(mouse_ids))
        mouse_palette = dict(zip(mouse_ids, palette))

        g = sns.FacetGrid(
            sub,
            col="genotype",
            col_order=genotype_order,
            height=5,
            aspect=1.1,
            sharey=True
        )

        g.map_dataframe(
            sns.barplot,
            x="week",
            y="mean_dFF",
            order=week_order,
            palette="Set2",
            errorbar="se",    # <-- correct usage in Seaborn 0.14+
            alpha=0.85
        )

        g.map_dataframe(
            sns.stripplot,
            x="week",
            y="mean_dFF",
            hue="mouse_id",
            palette=mouse_palette,
            dodge=True,
            alpha=0.8,
            size=5,
            jitter=True
        )

        for ax in g.axes.flatten():
            if ax.legend_:
                ax.legend_.remove()

        handles, labels = ax.get_legend_handles_labels()
        g.fig.legend(handles, labels, title="Mouse ID", bbox_to_anchor=(0.92, 0.85), frameon=False)

        g.set_axis_labels("Week", "Mean ΔF/F (%)")
        g.set_titles("{col_name}")
        g.fig.suptitle(
            f"Behavior: {beh} — ΔF/F (0–2 s post-event) across weeks",
            fontsize=16,
            y=1.05
        )

        for ax in g.axes.flatten():
            ax.grid(axis="y", alpha=0.3)
            ax.set_xlabel("")

        plt.tight_layout()
        plt.show()

# --------------------------
# RUN PLOT
# --------------------------
plot_behavior_facet_genotype_post2s(mean_summary_post2s_df, condition_filter="Novel")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_peak_windows(summary_df, condition_filter=None):
    df = summary_df.copy()

    if condition_filter:
        df = df[df["condition"] == condition_filter]

    df = df[df["behavior"] != "Other"]

    # Melt into long format
    df_long = df.melt(
        id_vars=["mouse_id", "genotype", "week", "behavior"],
        value_vars=["peak_BL", "peak_PRE", "peak_POST"],
        var_name="window",
        value_name="peak_zscore"
    )

    window_order = ["peak_BL", "peak_PRE", "peak_POST"]

    for beh in df_long["behavior"].unique():
        sub = df_long[df_long["behavior"] == beh]

        plt.figure(figsize=(6,5))

        sns.barplot(
            data=sub,
            x="window",
            y="peak_zscore",
            order=window_order,
            errorbar="se",
            palette="Set2"
        )

        sns.stripplot(
            data=sub,
            x="window",
            y="peak_zscore",
            hue="mouse_id",
            dodge=True,
            jitter=True,
            alpha=0.7
        )

        plt.title(f"{beh} — Peak z-scored ΔF/F")
        plt.ylabel("Peak z-score")
        plt.xlabel("Window")
        plt.legend(bbox_to_anchor=(1.05,1), frameon=False)
        plt.tight_layout()
        plt.show()

In [ ]:
summary_peak = []  # <-- THIS is what you're missing

for trial_key, trial_data in combined_trials.items():
    fp_info = fp_traces.get(trial_key, {})
    condition = fp_info.get("condition")
    week = fp_info.get("week")
    genotype = fp_info.get("genotype")
    mouse_id = fp_info.get("mouse_id", trial_key)

    if "dFF" not in trial_data.columns or "Behavior" not in trial_data.columns:
        continue

    signal = trial_data["dFF"].values
    behaviors = trial_data["Behavior"].values

    unique_behaviors = [b for b in np.unique(behaviors) if b != "Other"]

    for beh in unique_behaviors:

        starts = np.where(np.diff(np.r_[0, (behaviors == beh).astype(int)]) == 1)[0]
        if len(starts) == 0:
            continue

        segments = []
        for idx in starts:
            if idx >= pre_n and (idx + post_n) < len(signal):
                seg = signal[idx-pre_n:idx+post_n].copy()

                seg -= np.mean(seg[:pre_n])

                seg_std = np.std(seg)
                if seg_std == 0:
                    continue

                seg_z = seg / seg_std

                if len(seg_z) == expected_len:
                    segments.append(seg_z)

        if len(segments) == 0:
            continue

        mouse_mean_seg = np.mean(np.column_stack(segments), axis=1)

        peak_BL = np.max(mouse_mean_seg[BL_idx])
        peak_PRE = np.max(mouse_mean_seg[PRE_idx])
        peak_POST = np.max(mouse_mean_seg[POST_idx])

        summary_peak.append({
            "mouse_id": mouse_id,
            "genotype": genotype,
            "condition": condition,
            "week": week,
            "behavior": beh,
            "peak_BL": peak_BL,
            "peak_PRE": peak_PRE,
            "peak_POST": peak_POST,
            "n_events": len(segments)
        })

In [ ]:
summary_peak_df = pd.DataFrame(summary_peak)
# Run plot for a specific condition (e.g. Novel)
plot_peak_windows(summary_peak_df, condition_filter="Novel")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def plot_behavior_facet_genotype_with_lines(mean_df, condition_filter=None):
    """
    For each behavior (excluding 'Other'):
        • One figure with NE/WT facets
        • x-axis = week
        • bars = mean ± SEM
        • points = each mouse
        • lines = connect same mouse across weeks
    """

    df = mean_df.copy()

    # Filter specific condition (Social, Novel, Hab)
    if condition_filter is not None:
        df = df[df["condition"] == condition_filter]

    # Remove "Other"
    df = df[df["behavior"] != "Other"]

    if df.empty:
        print(f"[WARN] No data available after filtering for '{condition_filter}' and excluding 'Other'")
        return

    # Ordered weeks
    week_order = ["Preinduction", "W1", "W2", "W3"]
    df["week"] = pd.Categorical(df["week"], categories=week_order, ordered=True)

    # Ordered genotype facets
    genotype_order = ["NE", "WT"]
    df["genotype"] = pd.Categorical(df["genotype"], categories=genotype_order, ordered=True)

    # Behaviors to plot
    behaviors = sorted(df["behavior"].unique())

    for beh in behaviors:
        sub = df[df["behavior"] == beh]
        if sub.empty:
            continue

        # --- FACET SETUP ---
        g = sns.FacetGrid(
            sub,
            col="genotype",
            col_order=genotype_order,
            height=5,
            aspect=1.1,
            sharey=True
        )

        # --- BARPLOT (mean ± SEM) ---
        g.map_dataframe(
            sns.barplot,
            x="week",
            y="mean_dFF",
            order=week_order,
            palette="Set2",
            errorbar="se",
            alpha=0.8,
            width=0.65
        )

        # --- POINTS (each mouse) ---
        g.map_dataframe(
            sns.stripplot,
            x="week",
            y="mean_dFF",
            order=week_order,
            color="black",
            size=6,
            alpha=0.8,
            jitter=False
        )

        # --- LINES TRACKING EACH MOUSE ACROSS WEEKS ---
        def add_mouse_lines(data, color=None, **kwargs):
            """
            Draw lines connecting the same mouse across weeks inside each facet.
            """
            mice = data["mouse_id"].unique()
            for m in mice:
                df_m = data[data["mouse_id"] == m].sort_values("week")
                plt.plot(df_m["week"], df_m["mean_dFF"],
                         marker="o",
                         color="black",
                         alpha=0.5,
                         linewidth=1.2)

        g.map_dataframe(add_mouse_lines)

        # --- STYLE ---
        g.set_axis_labels("Week", "Mean ΔF/F (%)")
        g.set_titles("{col_name}")  # genotype as title (NE, WT)
        g.fig.suptitle(f"Behavior: {beh} — ΔF/F across weeks", fontsize=16, y=1.05)

        for ax in g.axes.flatten():
            ax.grid(axis="y", alpha=0.3)

        plt.tight_layout()
        plt.show()


In [ ]:
# # reloading traces later

# import pickle
# from pathlib import Path

# out_path = Path("fp_traces_precomputed.pkl")

# with open(out_path, "wb") as f:
#     pickle.dump(fp_traces, f)

# print(f"Saved fp_traces to {out_path.resolve()}")




In [ ]:

# import pickle

# with open("fp_traces_precomputed.pkl", "rb") as f:
#     fp_traces_reload = pickle.load(f)

# #print(fp_traces_reload)
# print(fp_traces)